# Triagegeist — Modeling, Red-Flag NLP & Equity Audit for ED Triage

**Thesis:** *Modeling, Reverse-Engineering, and Auditing an Emergency Triage Policy.*

---

## Clinical Framing

Emergency triage assigns every arriving patient a priority level — in the Emergency Severity Index
(ESI) framework used across North America and increasingly in Europe, **Level 1** is
*Resuscitation* (immediate life threat), and **Level 5** is *Non-urgent* (no resources needed).
The cardinal safety error is **undertriage**: assigning too low a priority to a critically ill
patient, delaying life-saving intervention. Studies using MIMIC-IV-ED and NHAMCS data document
undertriage rates exceeding 22% for elderly patients and systematic under-prioritisation of
patients with limited English proficiency, Black patients (aOR 0.76 for high-acuity triage), and
women with atypical cardiac presentations. A 1-level undertriage error in ESI L1–L2 is not
"off-by-one" — it is clinically dangerous.

**Three pillars, one integrated analysis:**

| Pillar | What we do | Why it matters |
|---|---|---|
| **A — Calibrated Acuity Model** | LightGBM 5-fold OOF, isotonic calibration, SHAP rule reverse-engineering, outcome validation | Reveals *what rule* the triage policy encodes; surfaces age-blindness as a contrast with real EDs |
| **B — NLP Red-Flag Flagger** | TF-IDF + LR/LGB on free-text chief complaints; 15-pattern lexicon; subjective/objective split | Independent safety net for presentations where vitals are deceptively normal |
| **C — Equity & Reliability Audit** | NEWS2-residual toolkit; negative control (null result); positive control (bias injection); inter-rater caterpillar plot; literature contrast | Reusable audit functions that *any hospital* can run on its own triage logs |

> **Data transparency note:** The dataset is entirely **synthetic** (no patient health information).
> The acuity label is near-deterministic in the **chief-complaint text**: Pillar A+ below shows that a
> one-line phrase lookup already scores about 0.996 and a leakage-safe text model about 0.9996, while a
> vitals-only view of the same label looks age-blind, demographically unbiased, and perfectly
> inter-rater consistent. We state this prominently because the findings contrast sharply with real ED
> literature, and because scientific honesty is the backbone of trustworthy clinical AI. External
> validation on MIMIC-IV-ED (~425 k stays) and NHAMCS (nationally representative) is the direct next
> step; those datasets share an identical triage schema and are fully accessible to the research community.

---
## Section 1 — Data & Methodology

### Dataset
Four CSV files under `/kaggle/input/competitions/triagegeist/`:

| File | Rows | Key columns |
|---|---|---|
| `train.csv` | 80,000 | 40 columns incl. `triage_acuity`, `disposition`, `ed_los_hours` |
| `test.csv` | 20,000 | 37 columns (no outcome cols) |
| `chief_complaints.csv` | 100,000 | `patient_id`, `chief_complaint_raw` (free text) |
| `patient_history.csv` | 100,000 | `patient_id` + 25 binary `hx_*` comorbidity flags |

Join key: `patient_id` (format `TG-XXXXXXXXX`); train/test patient IDs are **disjoint**.

### Leakage Protocol (NON-NEGOTIABLE)
`disposition` and `ed_los_hours` are **post-triage outcomes** absent from `test.csv`.
They are **never** used as model features. They appear only in the Pillar A outcome-validation
panel to demonstrate that predicted acuity tracks real clinical severity.

### Cross-validation
`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` on `triage_acuity`.
Class 1 is only ~4% of the training set; stratification is essential to preserve representation
of the safety-critical level in every fold.

## Data disclosure

All data comes from the competition's Triagegeist dataset, released by the Laitinen-Fredriksson Foundation under a non-commercial research license. The records are entirely synthetic; no real patient health information is involved. No external datasets are used anywhere in this notebook. We read four files from `/kaggle/input/competitions/triagegeist/`, all joined on `patient_id`:

| File | Rows | What it holds |
|---|---|---|
| `train.csv` | 80,000 | intake features (vitals, demographics, triage context, utilisation), the `triage_acuity` label, and the post-triage outcomes `disposition` and `ed_los_hours` |
| `test.csv` | 20,000 | the same intake features, without the label or outcomes |
| `chief_complaints.csv` | 100,000 | free-text `chief_complaint_raw` per patient |
| `patient_history.csv` | 100,000 | 25 binary `hx_*` comorbidity flags |
| `sample_submission.csv` | 20,000 | the required submission format |

`disposition` and `ed_los_hours` are post-triage outcomes absent from the test set; we use them only to validate predictions, never as model features. Pain score uses `-1` as a missing sentinel, which we convert to NaN. Vitals are missing not at random (recorded less often for lower-acuity patients), which we let LightGBM handle natively.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker

# ── Canonical constants ────────────────────────────────────────────────────
DATA         = "/kaggle/input/competitions/triagegeist/"
RANDOM_STATE = 42
LEAKAGE      = ["disposition", "ed_los_hours"]   # NEVER model features
TARGET       = "triage_acuity"

PROTECTED = ["language", "insurance_type", "age_group", "sex"]
VITALS    = [
    "systolic_bp", "diastolic_bp", "mean_arterial_pressure", "pulse_pressure",
    "heart_rate", "respiratory_rate", "temperature_c", "spo2",
    "gcs_total", "pain_score", "shock_index", "news2_score"
]

np.random.seed(RANDOM_STATE)

# ── Canonical load() / clean() ─────────────────────────────────────────────
def load():
    train = pd.read_csv(DATA + "train.csv")
    test  = pd.read_csv(DATA + "test.csv")
    cc    = pd.read_csv(DATA + "chief_complaints.csv")
    ph    = pd.read_csv(DATA + "patient_history.csv")
    cc = cc.drop(columns=["chief_complaint_system"], errors="ignore")
    train = train.merge(cc, on="patient_id", how="left").merge(ph, on="patient_id", how="left")
    test  = test.merge(cc,  on="patient_id", how="left").merge(ph, on="patient_id", how="left")
    return train, test


def clean(df):
    df = df.copy()
    df.loc[df["pain_score"] < 0,       "pain_score"]    = np.nan
    if "pulse_pressure" in df.columns:
        df.loc[df["pulse_pressure"] < 0, "pulse_pressure"] = np.nan
    return df


# ── Load once; reuse everywhere ────────────────────────────────────────────
print("Loading and joining tables …")
train_raw, test_raw = load()
train = clean(train_raw)
test  = clean(test_raw)

print(f"train shape : {train.shape}")
print(f"test  shape : {test.shape}")
print(f"\nTarget distribution (train):")
print(train[TARGET].value_counts().sort_index().rename("count").to_frame()
      .assign(pct=lambda d: (d["count"] / len(train) * 100).round(1)))

In [ ]:
# Target distribution bar chart
fig, ax = plt.subplots(figsize=(7, 4))
vc = train[TARGET].value_counts().sort_index()
colors_esi = ["#d62728", "#ff7f0e", "#2ca02c", "#1f77b4", "#9467bd"]
bars = ax.bar([f"L{i}" for i in vc.index], vc.values, color=colors_esi, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, vc.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f"{val:,}\n({val/len(train)*100:.1f}%)", ha="center", va="bottom", fontsize=9)
ax.set_xlabel("ESI Acuity Level (1 = most urgent)", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.set_title("Triage Acuity Distribution — Training Set (n=80,000)", fontsize=12)
ax.set_ylim(0, vc.max() * 1.18)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("target_distribution.png", dpi=120)
plt.show()
print("Note: class imbalance (L1 only 4%) → StratifiedKFold(5) mandatory.")

---
## Section 2 — Data Characterization

### The Physiological Structure of the Acuity Label

EDA reveals that `triage_acuity` is a near-deterministic function of vital signs.
Spearman ρ(NEWS2, acuity) = **−0.82**; the top ANOVA-F features are `news2_score`
(F=112,428), `gcs_total` (F=83,591), `spo2`, `shock_index`, `respiratory_rate`, and `pain_score`.
By contrast, `age` (F=0.53, p=0.71), `bmi`, `weight_kg`, and `height_cm` are statistically null —
the synthetic generator assigns acuity from physiology alone, ignoring demographics entirely.

This is the dataset's most important property: it is a **perfect negative control** for bias
testing (Section 5) and sets up the headline SHAP finding in Section 3.

### Leakage-Safe Outcome Validation Frame
`disposition` and `ed_los_hours` are never model inputs, but they are powerful validators:
predicted acuity should track admission rate, mortality, and length-of-stay monotonically.
The validation panel appears in Section 3 after the model is trained.

In [ ]:
from scipy import stats

# ── ANOVA F-rankings for key features ─────────────────────────────────────
anova_cols = VITALS + ["age", "bmi", "weight_kg", "height_cm",
                       "num_prior_ed_visits_12m", "num_comorbidities"]
anova_cols = [c for c in anova_cols if c in train.columns]

groups = [train.loc[train[TARGET] == k, anova_cols].values for k in sorted(train[TARGET].unique())]
f_stats = {}
for col in anova_cols:
    col_groups = [train.loc[train[TARGET] == k, col].dropna().values
                  for k in sorted(train[TARGET].unique())]
    try:
        f, p = stats.f_oneway(*col_groups)
        f_stats[col] = (float(f), float(p))
    except Exception:
        f_stats[col] = (0.0, 1.0)

f_df = (pd.DataFrame.from_dict(f_stats, orient="index", columns=["F_stat", "p_value"])
        .sort_values("F_stat", ascending=False))
print("ANOVA F-statistics vs triage_acuity (top features):")
print(f_df.round(1).to_string())

In [ ]:
# NEWS2 mean by acuity level
news2_by_acuity = train.groupby(TARGET)["news2_score"].mean().round(2)
print("Mean NEWS2 score by acuity level:")
for k, v in news2_by_acuity.items():
    print(f"  L{k}: {v:.2f}")

rho, pval = stats.spearmanr(train["news2_score"].fillna(train["news2_score"].median()),
                             train[TARGET])
print(f"\nSpearman ρ(NEWS2, acuity) = {rho:.4f}  p={pval:.2e}")
print("(Negative because L1=most urgent has HIGHEST NEWS2)")

# Plot: NEWS2 distribution by acuity level
fig, ax = plt.subplots(figsize=(9, 4))
for k, color in zip(sorted(train[TARGET].unique()), colors_esi):
    subset = train.loc[train[TARGET] == k, "news2_score"].dropna()
    ax.hist(subset, bins=30, alpha=0.55, label=f"L{k} (n={len(subset):,})",
            color=color, density=True)
ax.set_xlabel("NEWS2 Score", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title(f"NEWS2 Score Distribution by Acuity Level\n(Spearman ρ = {rho:.2f} — the acuity label is largely a NEWS2 function)",
             fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("news2_by_acuity.png", dpi=120)
plt.show()

---
## Section 3 — Pillar A: Calibrated Acuity Model

### Narrative

We train a LightGBM multiclass model to predict ESI-style triage acuity (1–5) from structured
intake — vitals, comorbidities, utilization, and chief-complaint system category — using
5-fold stratified cross-validation and **strictly excluding** `disposition` and `ed_los_hours`.

The model reaches **0.855 accuracy** and a **quadratic-weighted kappa of 0.930**, but the
accuracy is the least interesting number. Two findings matter more.

**Calibration:** A clinical decision-support tool must report trustworthy probabilities, not
just labels. Isotonic recalibration cuts expected calibration error from **ECE = 0.0067 to
0.0014**, so the model's confidence can be safely surfaced to a triage nurse.

**What the model learned (SHAP):** `gcs_total`, `pain_score`, and `news2_score` carry nearly
all signal. `age`, `BMI`, `weight`, and `height` carry essentially zero importance — the
dataset's implicit triage policy is a pure physiological function. That is reassuring for
fairness, but clinically incomplete: real geriatric patients are undertriaged precisely because
*compensated physiology hides severity*. The age-blindness of this synthetic policy is
the sharpest teaching contrast with real-world ED data.

**Outcome validation:** predicted acuity tracks real outcomes monotonically — admission falls
from 71% (L1) to 4% (L5), mortality from 8% to 0% — confirming the model captures genuine
severity, not a synthetic artefact.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    recall_score, confusion_matrix
)
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
import shap

# ── Feature matrix construction ────────────────────────────────────────────
DROP_ALWAYS = ["patient_id", TARGET] + LEAKAGE + ["chief_complaint_raw"]

def build_features(df):
    feature_cols = [c for c in df.columns if c not in DROP_ALWAYS]
    X = df[feature_cols].copy()
    for col in X.select_dtypes(include="object").columns:
        X[col] = X[col].astype("category")
    y = df[TARGET].values
    return X, y, feature_cols

X, y, feature_cols = build_features(train)
outcome_df = train[LEAKAGE].copy()   # reserved for outcome validation panel

cat_features = [c for c in X.columns if X[c].dtype.name == "category"]
print(f"Feature matrix: {X.shape[1]} columns")
print(f"Categorical features ({len(cat_features)}): {cat_features}")
print(f"Target distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

In [ ]:
# ── 5-fold LightGBM OOF training ──────────────────────────────────────────
n_classes   = 5
skf         = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_preds   = np.zeros(len(y), dtype=int)
oof_probs   = np.zeros((len(y), n_classes))
pa_fold_models = []

lgb_params = dict(
    objective        = "multiclass",
    num_class        = n_classes,
    num_leaves       = 127,
    learning_rate    = 0.05,
    n_estimators     = 500,
    min_child_samples= 20,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    random_state     = RANDOM_STATE,
    verbose          = -1,
    n_jobs           = -1,
)

print("Training LightGBM 5-fold OOF …")
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
        categorical_feature=cat_features,
    )

    proba = model.predict_proba(X_va)
    pred  = proba.argmax(axis=1) + 1

    oof_probs[va_idx] = proba
    oof_preds[va_idx] = pred
    pa_fold_models.append(model)

    fold_acc = accuracy_score(y_va, pred)
    print(f"  Fold {fold}: acc={fold_acc:.4f}  best_iter={model.best_iteration_}")

print("\nOOF training complete.")

In [ ]:
# ── OOF Metrics ───────────────────────────────────────────────────────────
pa_acc    = accuracy_score(y, oof_preds)
pa_mac_f1 = f1_score(y, oof_preds, average="macro")
pa_qwk    = cohen_kappa_score(y, oof_preds, weights="quadratic")
pa_per_cls= recall_score(y, oof_preds, average=None, labels=[1,2,3,4,5])
pa_cm     = confusion_matrix(y, oof_preds, labels=[1,2,3,4,5])

print("=" * 55)
print("  PILLAR A — OOF METRICS")
print("=" * 55)
print(f"  Accuracy     : {pa_acc:.4f}")
print(f"  Macro-F1     : {pa_mac_f1:.4f}")
print(f"  Quad. WK     : {pa_qwk:.4f}")
print(f"  Per-class recall:")
for i, r in enumerate(pa_per_cls):
    flag = "  ← safety-critical" if i < 2 else ""
    print(f"    L{i+1}: {r:.4f}{flag}")
print("=" * 55)

# Confusion matrix heatmap
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(pa_cm, cmap="Blues")
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels([f"Pred L{i}" for i in range(1,6)], fontsize=9)
ax.set_yticklabels([f"True L{i}" for i in range(1,6)], fontsize=9)
for r in range(5):
    for c in range(5):
        ax.text(c, r, str(pa_cm[r, c]), ha="center", va="center",
                color="white" if pa_cm[r,c] > pa_cm.max()*0.5 else "black", fontsize=8)
ax.set_title(f"OOF Confusion Matrix — Pillar A\n(acc={pa_acc:.4f}, QWK={pa_qwk:.4f})", fontsize=11)
ax.set_xlabel("Predicted acuity"); ax.set_ylabel("True acuity")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("pa_confusion_matrix.png", dpi=120)
plt.show()
print("Note: errors concentrate on adjacent low-acuity boundary (L3/L4/L5 — lowest clinical cost).")

In [ ]:
# ── ECE helper ────────────────────────────────────────────────────────────
def expected_calibration_error(y_true, probs, n_bins=10):
    # Multiclass ECE: average per-class binary ECE.
    n_classes = probs.shape[1]
    ece_per_class = []
    for k in range(n_classes):
        label_k = (y_true == (k + 1)).astype(float)
        prob_k  = probs[:, k]
        bins    = np.linspace(0, 1, n_bins + 1)
        ece_k   = 0.0
        for lo, hi in zip(bins[:-1], bins[1:]):
            mask = (prob_k >= lo) & (prob_k < hi)
            if mask.sum() == 0:
                continue
            ece_k += (mask.sum() / len(y_true)) * abs(label_k[mask].mean() - prob_k[mask].mean())
        ece_per_class.append(ece_k)
    return float(np.mean(ece_per_class))

# ── Isotonic calibration ───────────────────────────────────────────────────
pa_ece_before = expected_calibration_error(y, oof_probs)

oof_probs_cal = oof_probs.copy()
for k in range(n_classes):
    label_k = (y == (k + 1)).astype(float)
    ir = IsotonicRegression(out_of_bounds="clip")
    ir.fit(oof_probs[:, k], label_k)
    oof_probs_cal[:, k] = ir.transform(oof_probs[:, k])
row_sums = oof_probs_cal.sum(axis=1, keepdims=True)
row_sums = np.where(row_sums == 0, 1, row_sums)
oof_probs_cal /= row_sums

pa_ece_after = expected_calibration_error(y, oof_probs_cal)

print(f"ECE before calibration: {pa_ece_before:.5f}")
print(f"ECE after  calibration: {pa_ece_after:.5f}")
print(f"ECE improvement       : {pa_ece_before - pa_ece_after:.5f}")

In [ ]:
# ── Reliability diagrams ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.flatten()
colors_b = ["#d62728","#1f77b4","#2ca02c","#ff7f0e","#9467bd"]
colors_a = ["#ffaaaa","#aec7e8","#98df8a","#ffbb78","#c5b0d5"]

for k in range(n_classes):
    ax = axes[k]
    label_k = (y == (k + 1)).astype(float)
    frac_b, mean_b = calibration_curve(label_k, oof_probs[:, k],    n_bins=10, strategy="uniform")
    frac_a, mean_a = calibration_curve(label_k, oof_probs_cal[:, k], n_bins=10, strategy="uniform")
    ax.plot([0,1],[0,1], "k--", lw=1, label="Perfect")
    ax.plot(mean_b, frac_b, "o-", color=colors_b[k], label=f"Before (ECE={pa_ece_before:.4f})", lw=1.5)
    ax.plot(mean_a, frac_a, "s-", color=colors_a[k], label=f"After  (ECE={pa_ece_after:.4f})", lw=1.5)
    ax.set_title(f"Class {k+1}{'  ← safety' if k<2 else ''}", fontsize=10,
                 fontweight="bold" if k<2 else "normal")
    ax.set_xlabel("Mean predicted prob"); ax.set_ylabel("Fraction positives")
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

ax = axes[5]
ax.axis("off")
ax.text(0.1, 0.5,
    f"Calibration Summary\n\n"
    f"ECE before: {pa_ece_before:.5f}\n"
    f"ECE after:  {pa_ece_after:.5f}\n"
    f"Improvement: {pa_ece_before - pa_ece_after:.5f}\n\n"
    f"Method: per-class isotonic\n"
    f"regression on OOF probabilities",
    fontsize=11, va="center", family="monospace",
    bbox=dict(boxstyle="round,pad=0.5", fc="#f0f0f0"))
plt.suptitle("Reliability Diagrams — Before vs After Isotonic Calibration", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("pa_reliability_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# ── SHAP rule reverse-engineering ─────────────────────────────────────────
print("Computing SHAP values (fold-1 model, 2000-sample subset) …")

fold1_model = pa_fold_models[0]
shap_idx    = list(skf.split(X, y))[0][1]
shap_sample = np.random.default_rng(RANDOM_STATE).choice(
    shap_idx, size=min(2000, len(shap_idx)), replace=False)
X_shap = X.iloc[shap_sample]

explainer   = shap.TreeExplainer(fold1_model)
shap_values = explainer.shap_values(X_shap)

# Normalize to per-class list of (N, F) arrays
if isinstance(shap_values, list):
    shap_by_class = [np.asarray(sv) for sv in shap_values]
    mean_abs_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_by_class], axis=0)
else:
    _sv = np.asarray(shap_values)
    if _sv.ndim == 3:
        shap_by_class = [_sv[:, :, c] for c in range(_sv.shape[2])]
        mean_abs_shap = np.abs(_sv).mean(axis=(0, 2))
    else:
        shap_by_class = [_sv]
        mean_abs_shap = np.abs(_sv).mean(axis=0)

mean_abs_shap = np.asarray(mean_abs_shap).ravel()
shap_importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)

print("\nTop-20 features by mean |SHAP|:")
print(shap_importance.head(20).round(4).to_string())

age_bmi_cols = [c for c in ["age","bmi","weight_kg","height_cm"] if c in shap_importance.index]
print(f"\n*** Age/BMI/body-size features (should be ≈ 0 — age-blindness finding) ***")
for c in age_bmi_cols:
    print(f"  {c:20s}  mean |SHAP| = {shap_importance.get(c, 0.0):.6f}")

In [ ]:
# SHAP global importance bar chart
top25 = shap_importance.head(25)
fig, ax = plt.subplots(figsize=(9, 6))
colors_bar = ["#d62728" if f in VITALS else "#1f77b4" for f in top25.index]
ax.barh(top25.index[::-1], top25.values[::-1], color=colors_bar[::-1])
ax.set_xlabel("Mean |SHAP| (all classes)", fontsize=11)
ax.set_title(
    "Global Feature Importance — Pillar A SHAP\n"
    "(red = vital sign, blue = other feature)\n"
    "Age/BMI/weight/height ≈ 0 → physiology-only triage policy",
    fontsize=11)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("pa_shap_global.png", dpi=120)
plt.show()

pa_top3_global = list(shap_importance.head(3).index)
pa_top3_cls1   = list(
    pd.Series(np.abs(shap_by_class[0]).mean(axis=0), index=feature_cols)
    .sort_values(ascending=False).head(3).index)
print(f"Top-3 global SHAP drivers : {pa_top3_global}")
print(f"Top-3 class-1 SHAP drivers: {pa_top3_cls1}")
print(f"Age-blindness confirmed   : {all(shap_importance.get(c, 0) < 0.01 for c in age_bmi_cols)}")

In [ ]:
# ── Outcome validation panel ──────────────────────────────────────────────
print("Outcome validation (leakage-safe) …")

val_df = pd.DataFrame({
    "predicted_acuity": oof_preds,
    "disposition"     : outcome_df["disposition"].values,
    "ed_los_hours"    : outcome_df["ed_los_hours"].values,
})
val_df["admitted"] = val_df["disposition"].str.lower().str.contains(
    r"admit|inpatient", na=False).astype(float)
val_df["deceased"] = (val_df["disposition"].str.lower() == "deceased").astype(float)

ov_table = (val_df.groupby("predicted_acuity")
            .agg(n=("predicted_acuity","count"),
                 admit_rate=("admitted","mean"),
                 mortality=("deceased","mean"),
                 mean_los_hrs=("ed_los_hours","mean"))
            .round(4))
print(ov_table.to_string())

admit_mono = all(ov_table["admit_rate"].iloc[i] >= ov_table["admit_rate"].iloc[i+1]
                 for i in range(len(ov_table)-1))
los_mono   = all(ov_table["mean_los_hrs"].iloc[i] >= ov_table["mean_los_hrs"].iloc[i+1]
                 for i in range(len(ov_table)-1))
print(f"\nAdmit-rate monotone (L1→L5): {admit_mono}")
print(f"LOS monotone       (L1→L5): {los_mono}")

# Outcome validation figure
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
acuity_labels = [f"L{i}" for i in ov_table.index]
panel_colors  = ["#d62728","#ff7f0e","#2ca02c","#1f77b4","#9467bd"]

axes[0].bar(acuity_labels, ov_table["admit_rate"]*100, color=panel_colors)
axes[0].set_ylabel("Admission rate (%)"); axes[0].set_title("Admit Rate\nby Predicted Acuity")
axes[0].set_ylim(0, 100)
for i, v in enumerate(ov_table["admit_rate"]*100):
    axes[0].text(i, v+1.5, f"{v:.1f}%", ha="center", fontsize=9)

axes[1].bar(acuity_labels, ov_table["mortality"]*100, color=panel_colors)
axes[1].set_ylabel("Mortality rate (%)"); axes[1].set_title("Mortality Rate\nby Predicted Acuity")
axes[1].set_ylim(0, max(ov_table["mortality"].max()*130, 1))
for i, v in enumerate(ov_table["mortality"]*100):
    axes[1].text(i, v + ov_table["mortality"].max()*0.03, f"{v:.2f}%", ha="center", fontsize=9)

axes[2].bar(acuity_labels, ov_table["mean_los_hrs"], color=panel_colors)
axes[2].set_ylabel("Mean ED LOS (hours)"); axes[2].set_title("Mean LOS\nby Predicted Acuity")
for i, v in enumerate(ov_table["mean_los_hrs"]):
    axes[2].text(i, v+0.05, f"{v:.1f}h", ha="center", fontsize=9)

plt.suptitle(
    "Outcome Validation Panel — Predicted Acuity vs Real Outcomes\n"
    "(leakage-safe: disposition/ed_los_hours used only here, never as model features)",
    fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig("pa_outcome_validation.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Print Pillar A headline summary
print("=" * 60)
print("  PILLAR A — HEADLINE NUMBERS")
print("=" * 60)
print(f"  OOF Accuracy      : {pa_acc:.4f}  (target 0.855)")
print(f"  Macro-F1          : {pa_mac_f1:.4f}  (target 0.870)")
print(f"  Quadratic WK      : {pa_qwk:.4f}  (target 0.930)")
print(f"  Safety recall L1  : {pa_per_cls[0]:.4f}")
print(f"  Safety recall L2  : {pa_per_cls[1]:.4f}")
print(f"  ECE before→after  : {pa_ece_before:.5f} → {pa_ece_after:.5f}")
print(f"  Top-3 SHAP drivers: {pa_top3_global}")
print(f"  Age-blindness      : age/BMI/weight/height mean|SHAP|≈0")
print(f"  Outcome monotone   : admit={admit_mono}  LOS={los_mono}")
print("=" * 60)

---

## Pillar A+: the complaint-text acuity model, and a reality check

The Pillar-A model above deliberately **excludes** the free-text chief complaint and tops out near **0.855**. But a triage nurse always reads the complaint. Here we feed it back in through **in-fold TF-IDF**: the vocabulary is fit only on each fold's training split, so the validation rows never leak into the vectorizer. Then we stress-test the result honestly, because an accuracy this high deserves suspicion.

In [ ]:
# Complaint-text acuity model: physiology features + in-fold TF-IDF (word + char)
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold   # StratifiedKFold imported earlier

DROP_TXT = ["patient_id", TARGET] + LEAKAGE + ["chief_complaint_raw"]
feat_txt = [c for c in train.columns if c not in DROP_TXT]
base = train[feat_txt].copy(); cat_idx = []
for i, c in enumerate(feat_txt):
    if not pd.api.types.is_numeric_dtype(base[c]):
        base[c] = pd.Categorical(base[c]).codes      # -1 = NaN -> treated as missing
        cat_idx.append(i)
base_X = base.to_numpy(np.float32)
txt    = train["chief_complaint_raw"].fillna("").to_numpy()
yA     = train[TARGET].to_numpy()

def make_text_model():
    return lgb.LGBMClassifier(objective="multiclass", num_class=5, num_leaves=127,
        learning_rate=0.07, n_estimators=600, min_child_samples=20, subsample=0.8,
        colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1, n_jobs=-1)

skfA = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oofA = np.zeros(len(yA), dtype=int); oofPA = np.zeros((len(yA), 5))
print("Training complaint-text acuity model (5-fold, in-fold TF-IDF) ...")
for fold, (tr, va) in enumerate(skfA.split(base_X, yA), 1):
    wv = TfidfVectorizer(max_features=4000, min_df=3, ngram_range=(1, 2), sublinear_tf=True)
    cv = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=3000, min_df=3)
    Wt, Wv = wv.fit_transform(txt[tr]), wv.transform(txt[va])
    Ct, Cv = cv.fit_transform(txt[tr]), cv.transform(txt[va])
    Xt = sp.hstack([sp.csr_matrix(base_X[tr]), Wt, Ct]).tocsr()
    Xv = sp.hstack([sp.csr_matrix(base_X[va]), Wv, Cv]).tocsr()
    m = make_text_model(); m.fit(Xt, yA[tr], categorical_feature=cat_idx)
    p = m.predict_proba(Xv); oofPA[va] = p; oofA[va] = p.argmax(1) + 1
    print(f"  Fold {fold}: acc={accuracy_score(yA[va], oofA[va]):.4f}")

accA  = accuracy_score(yA, oofA)
qwkA  = cohen_kappa_score(yA, oofA, weights="quadratic")
recA  = recall_score(yA, oofA, average=None, labels=[1, 2, 3, 4, 5])
dangA = int(((yA <= 2) & (oofA > yA)).sum())
confA = oofPA.max(1)
eceA = 0.0
for lo in np.linspace(0, 1, 11)[:-1]:
    mk = (confA >= lo) & (confA < lo + 0.1)
    if mk.sum():
        eceA += mk.mean() * abs((oofA[mk] == yA[mk]).mean() - confA[mk].mean())

print(f"\nComplaint-text acuity model -- OOF accuracy {accA:.4f} | QWK {qwkA:.4f} | top-label ECE {eceA:.4f}")
print("  per-class recall:", "  ".join(f"L{i+1}={r:.3f}" for i, r in enumerate(recA)))
print(f"  dangerous L1/L2 undertriage: {dangA}   (vitals-only model: 566)")
print(f"  confidence when correct {confA[oofA==yA].mean():.4f}  vs  when wrong {confA[oofA!=yA].mean():.4f}")

In [ ]:
# Reality check: is ~1.0 accuracy real, leakage, or memorization?
# (1) a one-line phrase->majority lookup; (2) the same lookup on UNSEEN phrases
# (held out by complaint via GroupKFold); (3) the model on UNSEEN phrases.
def lookup_cv(splitter, groups=None):
    accs = []
    it = splitter.split(txt, yA, groups) if groups is not None else splitter.split(txt, yA)
    for trn, val in it:
        maj  = pd.Series(yA[trn], index=txt[trn]).groupby(level=0).agg(lambda s: s.value_counts().idxmax())
        glob = pd.Series(yA[trn]).value_counts().idxmax()
        pv   = pd.Series(txt[val]).map(maj).fillna(glob).to_numpy()
        accs.append(accuracy_score(yA[val], pv))
    return float(np.mean(accs))

acc_lookup_random = lookup_cv(StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE))
acc_lookup_unseen = lookup_cv(GroupKFold(5), groups=txt)

oof_grp = np.zeros(len(yA), dtype=int)
for trn, val in GroupKFold(5).split(base_X, yA, groups=txt):
    wv = TfidfVectorizer(max_features=4000, min_df=3, ngram_range=(1, 2), sublinear_tf=True)
    Wt, Wv = wv.fit_transform(txt[trn]), wv.transform(txt[val])
    Xt = sp.hstack([sp.csr_matrix(base_X[trn]), Wt]).tocsr()
    Xv = sp.hstack([sp.csr_matrix(base_X[val]), Wv]).tocsr()
    m = make_text_model(); m.fit(Xt, yA[trn], categorical_feature=cat_idx)
    oof_grp[val] = m.predict(Xv)
acc_model_unseen = accuracy_score(yA, oof_grp)

print("Reality check -- why the score is high, and why it is not a bug")
print(f"  no leakage: {len(feat_txt)} features; 'disposition' present? {'disposition' in feat_txt}; "
      f"'ed_los_hours' present? {'ed_los_hours' in feat_txt}")
print(f"  (1) phrase->majority LOOKUP, random folds   : {acc_lookup_random:.4f}   (task is ~a lookup)")
print(f"  (2) same LOOKUP, UNSEEN phrases (GroupKFold) : {acc_lookup_unseen:.4f}   (memorization collapses)")
print(f"  (3) MODEL on UNSEEN phrases (GroupKFold)     : {acc_model_unseen:.4f}   (real word-level generalization)")
print(f"      vitals-only baseline                    : 0.8551")

**How to read this.** The acuity label in this synthetic dataset is almost entirely a function of the complaint phrase. A one-line lookup already scores about 0.996 when phrases recur across folds, which is why a trained model reaches about 0.9996. That is not leakage (no outcome column is a feature) and not pure memorization: held out by *phrase*, a lookup collapses to about 0.36 while the model still scores about 0.998, because it learns the words themselves ("severe", "shock", "perforation"). Since 99.3% of test complaints also appear in training, this accuracy is legitimate for the competition, but it is a property of a text-deterministic generator. In a real ED the same complaint maps to many acuities and published triage models plateau near 0.70 to 0.80, so we report this as an upper bound, not a clinical result. The practical point: with the complaint text included, the policy turns out to be **chief-complaint-text-driven**, and the SHAP "pure physiology" picture from Pillar A holds only when that text is withheld.

---
## Section 4 — Pillar B: Chief-Complaint NLP Red-Flag Flagger

### Narrative

The Pillar B flagger operates on a single clinical premise: *vitals can lie, but words rarely do.*
Physiological deterioration manifests in vital signs, and on this synthetic dataset,
vitals-only prediction is already near-perfect (ROC-AUC ≈ 0.997). The marginal lift from
adding free-text chief complaints is small — a finding we report without apology, because
the honest framing is what a clinical team needs.

**Honest interpretation of AUC ≈ 1.0 for the text model:** This is a synthetic-data artefact.
The dataset generator embeds severity adjectives ("acute", "severe", "mild") directly into
`chief_complaint_raw` as a near-perfect proxy for the acuity label. In real-world chief
complaints ("chest pain"), no such encoding exists. In MIMIC-IV-ED, we would expect text-only
AUC ~0.65–0.75 and a marginal lift over vitals of +0.01–0.03.

The real-world argument for the NLP flagger lives where the vitals model fails: the elderly
patient whose compensated physiology masks a ruptured aortic aneurysm; the atypical ACS
presentation in a woman with normal borderline-vital signs; the septic patient whose SpO2 looks
"acceptable" while they verbally report rigors and confusion.

Our **15-pattern red-flag lexicon**, anchored in ESI L1–L2 clinical definitions, achieves
100% high-acuity concentration for 8 of 15 patterns. As a conservative standalone safety net
firing on 13.4% of presentations, it is a last-resort catch for the presentations where
failure to escalate carries the highest mortality.

In [ ]:
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, recall_score, precision_score
)

# ── Binary target: ESI L1 or L2 = high acuity ─────────────────────────────
pb_y = train[TARGET].isin([1, 2]).astype(int)
print(f"High-acuity positives: {pb_y.sum():,} / {len(pb_y):,}  ({pb_y.mean()*100:.1f}%)")

# ── TF-IDF feature matrix ──────────────────────────────────────────────────
texts = train["chief_complaint_raw"].fillna("").str.lower()

word_tfidf = TfidfVectorizer(analyzer="word",    ngram_range=(1,2),
                             max_features=8000, min_df=3, sublinear_tf=True)
char_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5),
                             max_features=8000, min_df=3, sublinear_tf=True)
X_word = word_tfidf.fit_transform(texts)
X_char = char_tfidf.fit_transform(texts)
X_text = hstack([X_word, X_char], format="csr")
print(f"TF-IDF matrix: {X_text.shape[0]:,} × {X_text.shape[1]:,}")

In [ ]:
# ── 5-fold OOF: LR + LightGBM on text-only ───────────────────────────────
pb_skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
pb_oof_lr  = np.zeros(len(pb_y))
pb_oof_lgb = np.zeros(len(pb_y))

print("Training LR + LightGBM on TF-IDF (text-only) …")
for fold, (tr_idx, va_idx) in enumerate(pb_skf.split(X_text, pb_y), 1):
    lr = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000,
                            solver="lbfgs", random_state=RANDOM_STATE)
    lr.fit(X_text[tr_idx], pb_y.iloc[tr_idx])
    pb_oof_lr[va_idx] = lr.predict_proba(X_text[va_idx])[:, 1]

    pos_w  = (pb_y.iloc[tr_idx] == 0).sum() / max((pb_y.iloc[tr_idx] == 1).sum(), 1)
    ds_tr  = lgb.Dataset(X_text[tr_idx], label=pb_y.iloc[tr_idx])
    ds_va  = lgb.Dataset(X_text[va_idx], label=pb_y.iloc[va_idx], reference=ds_tr)
    params = dict(objective="binary", metric="auc", learning_rate=0.05, num_leaves=63,
                  min_child_samples=20, feature_fraction=0.8, scale_pos_weight=pos_w,
                  verbosity=-1, random_state=RANDOM_STATE)
    cb = lgb.train(params, ds_tr, num_boost_round=300, valid_sets=[ds_va],
                   callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
    pb_oof_lgb[va_idx] = cb.predict(X_text[va_idx])

    lr_auc  = roc_auc_score(pb_y.iloc[va_idx], pb_oof_lr[va_idx])
    lgb_auc = roc_auc_score(pb_y.iloc[va_idx], pb_oof_lgb[va_idx])
    print(f"  Fold {fold}: LR AUC={lr_auc:.4f}  LGB AUC={lgb_auc:.4f}")

pb_oof_text  = pb_oof_lgb   # primary text model
text_lgb_auc = roc_auc_score(pb_y, pb_oof_lgb)
text_lgb_ap  = average_precision_score(pb_y, pb_oof_lgb)
text_lr_auc  = roc_auc_score(pb_y, pb_oof_lr)
print(f"\nText-only LGB OOF: ROC-AUC={text_lgb_auc:.4f}  PR-AUC={text_lgb_ap:.4f}")

In [ ]:
# ── Marginal lift: vitals-only vs combined ────────────────────────────────
X_vitals_arr    = train[VITALS].values.astype(np.float32)
X_vitals_sparse = csr_matrix(X_vitals_arr)
X_combined      = hstack([X_text, X_vitals_sparse], format="csr")

pb_oof_vitals   = np.zeros(len(pb_y))
pb_oof_combined = np.zeros(len(pb_y))

print("Training vitals-only and combined models …")
for fold, (tr_idx, va_idx) in enumerate(pb_skf.split(X_vitals_arr, pb_y), 1):
    pos_w = (pb_y.iloc[tr_idx] == 0).sum() / max((pb_y.iloc[tr_idx] == 1).sum(), 1)
    params_base = dict(objective="binary", metric="auc", learning_rate=0.05, num_leaves=63,
                       min_child_samples=20, feature_fraction=0.8, scale_pos_weight=pos_w,
                       verbosity=-1, random_state=RANDOM_STATE)

    dv_tr = lgb.Dataset(X_vitals_arr[tr_idx], label=pb_y.iloc[tr_idx])
    dv_va = lgb.Dataset(X_vitals_arr[va_idx], label=pb_y.iloc[va_idx], reference=dv_tr)
    mdl_v = lgb.train(params_base, dv_tr, num_boost_round=300, valid_sets=[dv_va],
                      callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
    pb_oof_vitals[va_idx] = mdl_v.predict(X_vitals_arr[va_idx])

    dc_tr = lgb.Dataset(X_combined[tr_idx], label=pb_y.iloc[tr_idx])
    dc_va = lgb.Dataset(X_combined[va_idx], label=pb_y.iloc[va_idx], reference=dc_tr)
    mdl_c = lgb.train(params_base, dc_tr, num_boost_round=300, valid_sets=[dc_va],
                      callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)])
    pb_oof_combined[va_idx] = mdl_c.predict(X_combined[va_idx])

vitals_auc  = roc_auc_score(pb_y, pb_oof_vitals)
vitals_ap   = average_precision_score(pb_y, pb_oof_vitals)
comb_auc    = roc_auc_score(pb_y, pb_oof_combined)
comb_ap     = average_precision_score(pb_y, pb_oof_combined)
text_lift_auc = comb_auc - vitals_auc
text_lift_ap  = comb_ap  - vitals_ap

print("\n  MARGINAL LIFT TABLE (HONEST NOTE: AUC≈1.0 is a synthetic-data artefact)")
print(f"  {'Model':<22} {'ROC-AUC':>9}  {'PR-AUC':>9}")
print(f"  {'Text-only (LGB)':<22} {text_lgb_auc:>9.4f}  {text_lgb_ap:>9.4f}")
print(f"  {'Vitals-only (LGB)':<22} {vitals_auc:>9.4f}  {vitals_ap:>9.4f}")
print(f"  {'Combined (LGB)':<22} {comb_auc:>9.4f}  {comb_ap:>9.4f}")
print(f"\n  Text lift over vitals: ΔROC-AUC={text_lift_auc:+.4f}  ΔPR-AUC={text_lift_ap:+.4f}")
print("  Real-world expectation: text lift ~+0.01–0.03 AUC (MIMIC-IV-ED benchmark)")

In [ ]:
# ── Precision-Recall curve ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
prec_c, rec_c, _ = precision_recall_curve(pb_y, pb_oof_text)
prc_v, rec_v, _  = precision_recall_curve(pb_y, pb_oof_vitals)
prc_cb, rec_cb, _= precision_recall_curve(pb_y, pb_oof_combined)

ax.plot(rec_c,  prec_c,  lw=2, color="#1f77b4",
        label=f"Text LGB  (PR-AUC={text_lgb_ap:.3f})")
ax.plot(rec_v,  prc_v,   lw=2, color="#ff7f0e", linestyle="--",
        label=f"Vitals LGB (PR-AUC={vitals_ap:.3f})")
ax.plot(rec_cb, prc_cb,  lw=2, color="#2ca02c", linestyle=":",
        label=f"Combined   (PR-AUC={comb_ap:.3f})")
ax.axhline(pb_y.mean(), color="gray", lw=1, linestyle="--",
           label=f"Baseline prevalence ({pb_y.mean():.3f})")
ax.set_xlabel("Recall", fontsize=12); ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Precision-Recall Curve — High Acuity (ESI 1–2)\nText vs Vitals vs Combined (5-fold OOF)\n"
             "* Note: AUC≈1.0 is a synthetic-data artefact — severity adjectives encode the label",
             fontsize=10)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("pb_pr_curve.png", dpi=120)
plt.show()

In [ ]:
# ── Red-flag lexicon ──────────────────────────────────────────────────────
RED_FLAGS = {
    "thunderclap_headache"  : r"thunderclap|worst.{0,10}headache|worst headache",
    "chest_pain_diaphoresis": r"chest.{0,20}(pain|pressure|tightness).{0,40}(sweat|diaphor)|diaphor.{0,40}chest",
    "chest_pain"            : r"\bchest\s+(pain|pressure|tightness|discomfort)\b",
    "stroke_deficit"        : r"\bstroke\b|facial\s+droop|arm\s+weak|slurred\s+speech|aphasia|hemiplegia|focal\s+deficit",
    "respiratory_distress"  : r"\brespiratory\s+(failure|arrest|distress)\b|can.t\s+breathe|unable\s+to\s+breathe",
    "anaphylaxis"           : r"\banaphyla|allergic\s+reaction\s+(severe|acute)|throat\s+(closing|swelling)",
    "sepsis_rigors"         : r"\bsepsis\b|septic\s+shock|\brigors\b|rigors\s+and\s+fever|shaking\s+chills",
    "suicidal_ideation"     : r"suicid|overdose|self.?harm|wants\s+to\s+die",
    "hemorrhage"            : r"\bhemorrhage\b|massive\s+bleed|coughing\s+blood|hemoptysis|rectal\s+bleed\s+heavy",
    "cardiac_arrest"        : r"cardiac\s+arrest|pulseless|unresponsive\s+and\s+pulseless|vf\b|v\.?fib",
    "acute_abdomen"         : r"acute\s+abdomen|board.?like\s+abdomen|rigid\s+abdomen|ruptured\s+aort",
    "meningeal_signs"       : r"\bmeningit|nuchal\s+rigidity|stiff\s+neck\s+and\s+fever|photophobia\s+and\s+headache",
    "diabetic_emergency"    : r"diabetic\s+keto|dka\b|hypoglycemi.{0,10}(unresponsive|altered|severe)",
    "eclampsia"             : r"\beclampsia\b|pre.?eclampsia\s+severe|seizure\s+in\s+pregnan",
    "aortic_dissection"     : r"aortic\s+(dissect|tear|rupture)|tearing\s+(chest|back)\s+pain",
}

texts_raw = train["chief_complaint_raw"].fillna("").str.lower()
flag_cols  = {name: texts_raw.str.contains(pat, regex=True, na=False).astype(int)
              for name, pat in RED_FLAGS.items()}
flag_df    = pd.DataFrame(flag_cols)
any_flag   = flag_df.any(axis=1).astype(int)

n_flagged   = any_flag.sum()
flag_prec   = precision_score(pb_y, any_flag, zero_division=0)
flag_recall = recall_score(pb_y, any_flag, zero_division=0)
flag_auc    = roc_auc_score(pb_y, any_flag) if any_flag.nunique() > 1 else 0.5

print(f"Total flagged: {n_flagged:,} / {len(any_flag):,}  ({n_flagged/len(any_flag)*100:.2f}%)")
print(f"Lexicon standalone: Precision={flag_prec:.3f}  Recall={flag_recall:.3f}  AUC={flag_auc:.4f}")
print("\nPer-flag breakdown:")
for fname, fcol in flag_cols.items():
    cnt = fcol.sum()
    if cnt == 0:
        print(f"  {fname:<30} n=0")
        continue
    ha_rate = pb_y[fcol == 1].mean()
    print(f"  {fname:<30} n={cnt:>5}  HA-rate={ha_rate:.3f}")

In [ ]:
# Red-flag summary figure
flag_summary = [{"flag": n.replace("_"," "), "count": c.sum(),
                 "ha_rate": pb_y[c==1].mean() if c.sum()>0 else 0.0}
                for n, c in flag_cols.items()]
flag_summary_df = pd.DataFrame(flag_summary).sort_values("count", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].barh(flag_summary_df["flag"], flag_summary_df["count"], color="#4c72b0", height=0.65)
axes[0].set_xlabel("Count in training set", fontsize=11)
axes[0].set_title("Red-Flag Phrase Counts", fontsize=12)
axes[0].grid(axis="x", alpha=0.3)

colors_ha = ["#d62728" if r>=0.5 else "#ff9896" if r>=0.25 else "#aec7e8"
             for r in flag_summary_df["ha_rate"]]
axes[1].barh(flag_summary_df["flag"], flag_summary_df["ha_rate"], color=colors_ha, height=0.65)
axes[1].axvline(pb_y.mean(), color="black", lw=1.5, linestyle="--",
                label=f"Base rate ({pb_y.mean():.2f})")
axes[1].set_xlabel("High-Acuity Rate (actual)", fontsize=11)
axes[1].set_title("Red-Flag Pattern → High-Acuity Rate", fontsize=12)
axes[1].legend(fontsize=9)
axes[1].xaxis.set_major_formatter(ticker.PercentFormatter(1.0))
axes[1].grid(axis="x", alpha=0.3)

fig.suptitle("Red-Flag Lexicon: Prevalence and High-Acuity Concentration\n"
             "(red ≥50% precision, pink ≥25%, blue <25%)", fontsize=11, y=1.01)
fig.tight_layout()
plt.savefig("pb_lexicon_flags.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# ── Subjective vs objective complaint split ───────────────────────────────
SUBJ_PATTERN = (
    r"\bpain\b|chest\s+pain|dyspnea|shortness\s+of\s+breath|sob\b|dizziness|dizzy|"
    r"nausea|vomiting|palpitation|fatigue|weakness|syncope|confusion|"
    r"headache|abdominal\s+pain|back\s+pain|generalized\s+pain|malaise"
)
OBJ_PATTERN  = (
    r"\btrauma\b|laceration|fracture|burn\b|burn\s+injury|dislocation|"
    r"mva\b|motor\s+vehicle|fall\s+from|head\s+injury|blunt|wound\b|stabbing|"
    r"penetrating|crush\s+injury|amputation"
)
is_subj = texts_raw.str.contains(SUBJ_PATTERN, regex=True, na=False)
is_obj  = texts_raw.str.contains(OBJ_PATTERN,  regex=True, na=False)
complaint_type = pd.Series("other", index=train.index)
complaint_type[is_subj] = "subjective"
complaint_type[is_obj]  = "objective"

print("Subjective vs Objective split:")
print(complaint_type.value_counts())
print()
for ctype in ["subjective","objective","other"]:
    mask = complaint_type == ctype
    if mask.sum() < 50: continue
    auc_t = roc_auc_score(pb_y[mask], pb_oof_text[mask])
    lx_rec= recall_score(pb_y[mask], any_flag[mask], zero_division=0)
    lx_pre= precision_score(pb_y[mask], any_flag[mask], zero_division=0)
    print(f"  {ctype.upper():12s} n={mask.sum():>6}  HA-rate={pb_y[mask].mean():.3f}  "
          f"text-AUC={auc_t:.4f}  lex-recall={lx_rec:.3f}  lex-prec={lx_pre:.3f}")

print("\nKey finding: Subjective complaints (chest pain, dyspnea, pain) carry the most nurse")
print("discretion — and the most bias risk in real EDs. The lexicon's lower precision here (34.7%)")
print("reflects real-world challenge: severity is carried by adjectives, not the complaint noun.")

---
## Section 5 — Pillar C: Triage Equity & Reliability Audit

### Narrative

Emergency department triage is a high-stakes, time-pressured human decision. The literature
documents systematic disparities: patients with limited English proficiency are admitted at
higher rates than their triage level predicts (OR 1.16, PMC12208044), Black patients receive
lower-acuity triage despite equivalent physiological severity (aOR 0.76, arXiv 2503.22781),
and elderly patients face undertriage rates exceeding 22% (PMC4143318) — more than double the
accepted 10% threshold. These disparities concentrate in subjective complaint categories where
nurse discretion is highest.

We built a **reusable audit toolkit** — four Python functions with clean public interfaces —
that any hospital can apply to its own triage logs. The core metric is the **NEWS2 residual**:
the gap between a patient's assigned acuity and the acuity expected from their vital-sign
profile alone. A systematically positive residual for a subgroup indicates that group is
receiving *lower* acuity than their physiology predicts — the operational definition of
undertriage.

**Negative control (provided data):** The audit correctly produces null results across all
four protected attributes. Every group residual falls within ±0.019 acuity units; every
95% bootstrap CI straddles zero. This validates that the toolkit does not manufacture false
positives.

**Positive control (injected bias):** Synthetically injecting 1-level undertriage into just
5% of the Finnish-speaking cohort is reliably detected at 95% CI (effect size = 0.05, measured
residual +0.019, CI 0.013–0.025). Below that threshold the CI straddles zero — the audit is
calibrated, not trigger-happy.

**Inter-rater reliability:** Across 50 nurses the per-nurse residual standard deviation is
0.014 with range [−0.022, +0.035]. One nurse (NURSE-0013, residual +0.035, CI 0.006–0.064)
is flagged as a systematic outlier with a "cold" (under-triage) tendency.

In [ ]:
# ── Audit toolkit constants ───────────────────────────────────────────────
N_BOOT    = 2000
BOOT_SEED = 42

def news2_expected_acuity(df):
    # Fit monotone NEWS2 -> expected_acuity mapping; return expected values.
    # Uses NEWS2 bins (clinical thresholds); fitted on full provided df.
    df = df.copy()
    bins   = [-1, 0, 2, 4, 6, 8, 999]
    labels = [0,  1, 2, 3, 4,  5]
    df["_news2_bin"] = pd.cut(
        df["news2_score"].fillna(df["news2_score"].median()),
        bins=bins, labels=labels).astype(int)
    bin_mean    = df.groupby("_news2_bin")[TARGET].mean().sort_index()
    global_mean = df[TARGET].mean()
    return df["_news2_bin"].map(bin_mean).fillna(global_mean)


def _bootstrap_ci(values, stat_fn=np.mean, n_boot=N_BOOT, seed=BOOT_SEED, ci=0.95):
    # Bootstrap CI for a scalar statistic.
    rng   = np.random.default_rng(seed)
    boots = np.array([stat_fn(rng.choice(values, size=len(values), replace=True))
                      for _ in range(n_boot)])
    alpha = (1 - ci) / 2
    return float(np.quantile(boots, alpha)), float(np.quantile(boots, 1 - alpha))


def audit_by_group(df, attr):
    # Per-group equity audit for a single protected attribute.
    # Returns DataFrame with mean_residual, 95% CI, ci_straddles_0.
    df = df.copy()
    df["_expected"] = news2_expected_acuity(df)
    df["_residual"] = df[TARGET] - df["_expected"]
    global_mean_acuity = df[TARGET].mean()
    rows = []
    for grp, sub in df.groupby(attr, observed=True):
        resid_vals = sub["_residual"].dropna().values
        ci_lo, ci_hi = _bootstrap_ci(resid_vals)
        rows.append({
            "group"          : grp,
            "n"              : len(sub),
            "mean_acuity"    : float(sub[TARGET].mean()),
            "high_acuity_rate": float((sub[TARGET] <= 2).mean()),
            "mean_residual"  : float(np.mean(resid_vals)),
            "obs_vs_exp"     : float(sub[TARGET].mean() / global_mean_acuity),
            "ci_low"         : ci_lo,
            "ci_high"        : ci_hi,
            "ci_straddles_0" : bool(ci_lo <= 0.0 <= ci_hi),
        })
    return pd.DataFrame(rows).sort_values("group").reset_index(drop=True)

print("Audit toolkit functions defined.")
print(f"Protected attributes: {PROTECTED}")

In [ ]:
# ── NEGATIVE CONTROL: equity audit on provided data ──────────────────────
print("=" * 65)
print("NEGATIVE CONTROL — Equity Audit on Provided Synthetic Data")
print("=" * 65)

pc_audit_tables = {}
for attr in PROTECTED:
    tbl = audit_by_group(train, attr)
    pc_audit_tables[attr] = tbl
    max_abs    = tbl["mean_residual"].abs().max()
    all_stride = tbl["ci_straddles_0"].all()
    print(f"\n  Attribute: {attr}  (max|residual|={max_abs:.4f}  all_CI∋0={all_stride})")
    print(tbl[["group","n","mean_residual","ci_low","ci_high","ci_straddles_0"]].to_string(index=False))

all_null = all(tbl["ci_straddles_0"].all() for tbl in pc_audit_tables.values())
print(f"\n→ All CIs straddle zero? {all_null}")
print("  Expected True — synthetic data assigns acuity from physiology only.")

In [ ]:
# Forest plot — negative control
palette_c = {"language":"#4C72B0","insurance_type":"#DD8452",
             "age_group":"#55A868","sex":"#C44E52"}

fig, axes = plt.subplots(1, len(PROTECTED), figsize=(16, 6), sharey=False)
fig.suptitle("Equity Audit — Negative Control\n"
             "NEWS2-Residual (Mean ± 95% Bootstrap CI) by Protected Attribute",
             fontsize=12, fontweight="bold", y=1.02)

for ax, attr in zip(axes, PROTECTED):
    tbl    = pc_audit_tables[attr]
    groups = tbl["group"].astype(str).tolist()
    y_pos  = np.arange(len(groups))
    means  = tbl["mean_residual"].values
    lo_err = means - tbl["ci_low"].values
    hi_err = tbl["ci_high"].values - means
    ax.errorbar(means, y_pos, xerr=[lo_err, hi_err],
                fmt="o", color=palette_c[attr],
                ecolor=palette_c[attr], elinewidth=1.8,
                capsize=4, capthick=1.5, markersize=7)
    ax.axvline(0, color="black", linewidth=1.0, linestyle="--")
    ax.set_yticks(y_pos); ax.set_yticklabels(groups, fontsize=9)
    ax.set_xlabel("NEWS2-Residual\n(+ = under-triaged vs vitals)", fontsize=9)
    ax.set_title(attr.replace("_"," ").title(), fontsize=11, fontweight="bold")
    ax.grid(axis="x", alpha=0.3)
    for i, (m, _, hi) in enumerate(zip(means, tbl["ci_low"].values, tbl["ci_high"].values)):
        ax.text(hi + 0.003, i, f"{m:+.3f}", va="center", fontsize=7, color=palette_c[attr])

fig.tight_layout()
plt.savefig("pc_forest_negative.png", dpi=150, bbox_inches="tight")
plt.show()
print("All error bars cross zero — no detectable disparity. This is the expected and correct result.")

In [ ]:
# ── POSITIVE CONTROL: bias injection sensitivity ──────────────────────────
def inject_undertriage(df, attr, group, frac, delta):
    df = df.copy()
    mask  = df[attr] == group
    n_grp = mask.sum()
    n_inj = int(frac * n_grp)
    if n_inj == 0: return df
    rng    = np.random.default_rng(BOOT_SEED)
    chosen = rng.choice(df.index[mask].tolist(), size=n_inj, replace=False)
    df.loc[chosen, TARGET] = (df.loc[chosen, TARGET] + delta).clip(upper=5).astype(int)
    return df


# Auto-select largest language group as target
pc_sweep_attr  = "language"
pc_sweep_group = train[pc_sweep_attr].value_counts().index[0]
print(f"Positive control target: {pc_sweep_attr}=={pc_sweep_group!r}")

fracs  = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]
deltas = [0.25, 0.50, 0.75, 1.0,  1.5,  2.0]
records = []
for frac in fracs:
    for delta in deltas:
        df_inj  = inject_undertriage(train, pc_sweep_attr, pc_sweep_group, frac, delta)
        tbl     = audit_by_group(df_inj, pc_sweep_attr)
        row     = tbl[tbl["group"] == pc_sweep_group].iloc[0]
        detected= not row["ci_straddles_0"]
        records.append({"frac": frac, "delta": delta,
                         "effect_size": round(frac*delta,4),
                         "measured_residual": round(float(row["mean_residual"]),5),
                         "ci_low": round(float(row["ci_low"]),5),
                         "ci_high": round(float(row["ci_high"]),5),
                         "detected": detected})

pc_sweep_df = pd.DataFrame(records)
detected_rows = pc_sweep_df[pc_sweep_df["detected"]]
pc_detect_thresh = float(detected_rows["effect_size"].min()) if not detected_rows.empty else float("nan")
print(pc_sweep_df.to_string(index=False))
print(f"\nDetection threshold (min effect_size): {pc_detect_thresh:.3f}")

In [ ]:
# Sensitivity figure (effect-size scatter + detection heatmap)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f"Positive-Control Sensitivity — Injected Undertriage in {pc_sweep_attr}=={pc_sweep_group!r}\n"
    "How small an injected bias can the audit reliably detect?",
    fontsize=11, fontweight="bold")

unique_effects = pc_sweep_df.sort_values("effect_size")
colors_det = ["#d62728" if not d else "#2ca02c" for d in unique_effects["detected"]]
ax1.scatter(unique_effects["effect_size"], unique_effects["measured_residual"],
            c=colors_det, zorder=3, s=60, edgecolors="k", linewidths=0.5)
ax1.vlines(unique_effects["effect_size"], unique_effects["ci_low"], unique_effects["ci_high"],
           color=colors_det, alpha=0.3, linewidth=2)
ax1.axhline(0, color="black", linestyle="--", linewidth=1.0)
if not np.isnan(pc_detect_thresh):
    ax1.axvline(pc_detect_thresh, color="#ff7f0e", linestyle=":", linewidth=2,
                label=f"Detection threshold ≈ {pc_detect_thresh:.2f}")
ax1.set_xlabel("Injected Effect Size (frac × delta)", fontsize=10)
ax1.set_ylabel("Measured NEWS2-Residual ± 95% CI", fontsize=10)
ax1.set_title("Residual vs. Injected Effect", fontsize=11)
green_p = mpatches.Patch(color="#2ca02c", label="Detected (CI excl. 0)")
red_p   = mpatches.Patch(color="#d62728", label="Not detected")
ax1.legend(handles=[green_p, red_p], fontsize=8, loc="upper left")
ax1.grid(alpha=0.3)

fracs_u  = sorted(pc_sweep_df["frac"].unique())
deltas_u = sorted(pc_sweep_df["delta"].unique())
Z = np.zeros((len(fracs_u), len(deltas_u)))
for i, f in enumerate(fracs_u):
    for j, d in enumerate(deltas_u):
        row = pc_sweep_df[(pc_sweep_df["frac"]==f)&(pc_sweep_df["delta"]==d)]
        if not row.empty: Z[i,j] = 1.0 if row.iloc[0]["detected"] else 0.0

im = ax2.imshow(Z, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto", origin="lower")
ax2.set_xticks(range(len(deltas_u)))
ax2.set_xticklabels([f"{d:.2f}" for d in deltas_u], fontsize=9)
ax2.set_yticks(range(len(fracs_u)))
ax2.set_yticklabels([f"{f:.0%}" for f in fracs_u], fontsize=9)
ax2.set_xlabel("Acuity Bump (delta)", fontsize=10)
ax2.set_ylabel("Fraction Affected (frac)", fontsize=10)
ax2.set_title("Detection Heatmap\n(Green = Detected at 95% CI)", fontsize=11)
for i in range(len(fracs_u)):
    for j in range(len(deltas_u)):
        ax2.text(j, i, "✓" if Z[i,j] else "✗",
                 ha="center", va="center", fontsize=12, color="black")
plt.colorbar(im, ax=ax2, label="Detected (1=yes)")
fig.tight_layout()
plt.savefig("pc_sensitivity_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Inter-rater reliability ───────────────────────────────────────────────
def audit_inter_rater(df):
    df = df.copy()
    df["_expected"] = news2_expected_acuity(df)
    df["_residual"] = df[TARGET] - df["_expected"]
    global_mean_resid = float(df["_residual"].mean())
    rows = []
    for nurse_id, sub in df.groupby("triage_nurse_id"):
        resid_vals = sub["_residual"].dropna().values
        ci_lo, ci_hi = _bootstrap_ci(resid_vals)
        is_outlier = not (ci_lo <= global_mean_resid <= ci_hi)
        rows.append({
            "nurse_id"     : nurse_id,
            "n_patients"   : len(sub),
            "mean_residual": float(np.mean(resid_vals)),
            "ci_low"       : ci_lo,
            "ci_high"      : ci_hi,
            "outlier"      : is_outlier,
            "tendency"     : (
                "cold (under-triage)" if np.mean(resid_vals) > global_mean_resid + 0.02
                else "hot (over-triage)"  if np.mean(resid_vals) < global_mean_resid - 0.02
                else "normal"),
        })
    return pd.DataFrame(rows).sort_values("mean_residual", ascending=False).reset_index(drop=True), global_mean_resid

nurse_df, pc_global_mean_resid = audit_inter_rater(train)
n_outliers = int(nurse_df["outlier"].sum())
print(f"Nurses analyzed    : {len(nurse_df)}")
print(f"Global mean resid  : {pc_global_mean_resid:.4f}")
print(f"Per-nurse std      : {nurse_df['mean_residual'].std():.4f}")
print(f"Range              : [{nurse_df['mean_residual'].min():.4f}, {nurse_df['mean_residual'].max():.4f}]")
print(f"Outlier nurses     : {n_outliers}")
if n_outliers > 0:
    print("\nOutlier nurses:")
    print(nurse_df[nurse_df["outlier"]][
        ["nurse_id","n_patients","mean_residual","ci_low","ci_high","tendency"]
    ].to_string(index=False))

In [ ]:
# Caterpillar plot — inter-rater
n_nurses = len(nurse_df)
y_pos    = np.arange(n_nurses)
means_n  = nurse_df["mean_residual"].values
lo_err_n = means_n - nurse_df["ci_low"].values
hi_err_n = nurse_df["ci_high"].values - means_n
colors_n = ["#d62728" if o else "#4C72B0" for o in nurse_df["outlier"]]

fig, ax = plt.subplots(figsize=(5, max(6, n_nurses * 0.25)))
for i, (m, le, he, col) in enumerate(zip(means_n, lo_err_n, hi_err_n, colors_n)):
    ax.errorbar(m, i, xerr=[[le],[he]], fmt="none",
                ecolor=col, elinewidth=1.2, capsize=2, capthick=1.0, alpha=0.8)
ax.scatter(means_n, y_pos, c=colors_n, s=20, zorder=3)
ax.axvline(pc_global_mean_resid, color="black", linewidth=1.0, linestyle="--",
           label=f"Global mean ({pc_global_mean_resid:+.4f})")
ax.axvline(pc_global_mean_resid+0.02, color="gray", linewidth=0.7, linestyle=":", alpha=0.7)
ax.axvline(pc_global_mean_resid-0.02, color="gray", linewidth=0.7, linestyle=":", alpha=0.7)
for pos_i, (_, row) in enumerate(nurse_df.iterrows()):
    if row["outlier"]:
        ax.text(row["ci_high"]+0.001, pos_i, str(row["nurse_id"]),
                va="center", fontsize=7, color="#d62728")
ax.set_yticks([])
ax.set_xlabel("Per-Nurse NEWS2-Residual (Mean ± 95% CI)", fontsize=10)
ax.set_title(f"Inter-Rater Reliability — {n_nurses} Nurses\n"
             "Caterpillar Plot (Red = Outlier vs. Global Mean)", fontsize=11, fontweight="bold")
normal_p  = mpatches.Patch(color="#4C72B0", label="Normal tendency")
outlier_p = mpatches.Patch(color="#d62728", label="Outlier nurse")
ax.legend(handles=[normal_p, outlier_p], fontsize=9)
ax.grid(axis="x", alpha=0.3)
ax.text(0.98, 0.02, "Real-world ESI κ ≈ 0.6–0.9\nThis synthetic data: near-perfect",
        transform=ax.transAxes, fontsize=7, ha="right", va="bottom", color="gray", style="italic")
plt.tight_layout()
plt.savefig("pc_caterpillar_nurses.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Literature contrast table ─────────────────────────────────────────────
lit_rows = [
    {"Attribute":        "Language (LEP)",
     "Real-World Effect": "OR 1.16 admission (Spanish/Chinese vs English); +50-91 min wait at low acuity",
     "Source":            "PMC12208044 (n=58,079)"},
    {"Attribute":        "Insurance type",
     "Real-World Effect": "Uninsured/Medicaid receive lower-acuity triage (NHAMCS population data)",
     "Source":            "NHAMCS public-use files"},
    {"Attribute":        "Age group (elderly)",
     "Real-World Effect": ">22% undertriage rate; OR 1.49 for age>=65",
     "Source":            "PMC4143318, PMC10890089"},
    {"Attribute":        "Sex",
     "Real-World Effect": "Men aOR 1.16 high-acuity triage vs women; cardiac under-recognition in women",
     "Source":            "arXiv 2503.22781 (n=297,355)"},
]

attr_key_map = {
    "Language (LEP)": "language", "Insurance type": "insurance_type",
    "Age group (elderly)": "age_group", "Sex": "sex"
}
for row in lit_rows:
    attr = attr_key_map[row["Attribute"]]
    tbl  = pc_audit_tables[attr]
    max_abs    = tbl["mean_residual"].abs().max()
    all_stride = tbl["ci_straddles_0"].all()
    row["Our max|residual|"] = f"{max_abs:.4f}"
    row["All CI contain 0?"] = str(all_stride)
    row["Verdict"] = (
        "NULL — physiology-only assignment" if all_stride and max_abs < 0.02
        else "WEAK" if max_abs < 0.05 else "DETECTED")

lit_df = pd.DataFrame(lit_rows)
print("=" * 90)
print("LITERATURE CONTRAST TABLE")
print("=" * 90)
print(lit_df[["Attribute","Our max|residual|","All CI contain 0?","Verdict"]].to_string(index=False))
print()
print("The gap between real-world effect sizes (OR 0.76-1.49) and our null audit result is")
print("itself informative: the synthetic generator assigns acuity from physiology only - the ideal")
print("that real triage should aspire to but does not achieve.")

---
## Section 6 — Pillar D: Outcome-Anchored Undertriage Detection & Second Opinion

### Narrative

The cornerstone limitation of standard triage validation is circularity: if you train a model
on the triage acuity label to predict outcomes, you are largely predicting a label that was
already derived from the same vital signs. Pillar D breaks this cycle. We deliberately exclude
`triage_acuity` from the feature matrix and train a LightGBM binary classifier to predict a
hard real-world outcome — whether a patient is admitted, transferred, or dies — directly from
triage-time features (vitals, demographics, comorbidities, utilisation). This produces an
**independent outcome risk score** that the acuity policy cannot trivially reproduce.

The model achieves ROC-AUC 0.813 and PR-AUC 0.729 — genuinely non-trivial given the 37.8%
base rate and the inherent noise in disposition decisions. This is the realistic range for
outcome prediction from triage features; models claiming AUC > 0.95 on such problems are
almost certainly using acuity or other post-triage information as features. Calibration is
good (ECE 0.014 before, improved to near-zero after isotonic regression), and the model stays
well-calibrated within all eight language subgroups and four age groups (ECE 0.001–0.015),
confirming it does not systematically mislead for minority-language or elderly patients.

The second contribution is **outcome-anchored undertriage detection**. Among the 34,418
patients assigned low-priority acuity levels 4 or 5, we flag the top-decile by predicted
outcome risk (threshold: 27.9% calibrated probability). This yields 3,646 undertriage
candidates — patients whose vital-sign and comorbidity profile suggests a substantially higher
risk of critical outcome than their assigned acuity implies. Validation shows flagged patients
have an 11.9% actual critical-outcome rate versus 9.3% for non-flagged low-acuity patients.
Case D illustrates the clinical value: "mild chest discomfort" was assigned L4, but the model
predicted 59.5% critical risk and the patient was ultimately admitted.

The equity audit confirms that flags are distributed equitably across language, insurance, age
and sex groups (all bootstrap CIs straddle the overall flag rate of 10.6%), consistent with
the null result established in Pillar C. Data forensics independently confirm the three derived
vitals (MAP, pulse pressure, shock index) are exact analytical formulae — R² ≥ 0.9999.

The `triage_second_opinion()` function provides a deployable decision-support interface: given
a patient row, it returns an assigned-acuity, calibrated outcome risk percent, risk tier,
red-flag text hits, and an undertriage warning with plain-language reasons — reusable on any
dataset sharing the same triage schema.

In [ ]:
# ── Pillar D: Outcome-Anchored Undertriage Detection ──────────────────────
# All variables prefixed d_ to avoid namespace collisions with Pillars A/B/C.
# Data is reloaded fresh (pd.py runs independently in ~32s on Kaggle).

import re as _re
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve

print("=" * 70)
print("Pillar D — Outcome-Anchored Undertriage Detection")
print("Seed: 42  |  triage_acuity EXCLUDED from outcome model features")
print("=" * 70)

# Reload data fresh (ensures no state dependency on earlier sections)
print("\n[D.1] Loading data for Pillar D …")
d_train_raw, _ = load()
d_train = clean(d_train_raw)
print(f"  train shape: {d_train.shape}")

# Store outcome columns (NEVER enter feature matrix)
d_outcome_raw = d_train["disposition"].copy()
d_acuity_true = d_train[TARGET].copy()

# ── Feature matrix: exclude patient_id, triage_acuity, disposition, ed_los_hours, raw text ──
D_EXCLUDE = ["patient_id", TARGET, "chief_complaint_raw"] + LEAKAGE
d_feature_cols = [c for c in d_train.columns if c not in D_EXCLUDE]
d_X = d_train[d_feature_cols].copy()
for _col in d_X.select_dtypes(include="object").columns:
    d_X[_col] = d_X[_col].astype("category")
d_cat_features = [c for c in d_X.columns if d_X[c].dtype.name == "category"]
print(f"  Feature matrix: {d_X.shape[1]} columns (triage_acuity excluded)")

# ── Binary outcome target: critical = admitted | transferred | deceased ────
D_CRITICAL = {"admitted", "transferred", "deceased"}
d_y_crit = d_outcome_raw.str.lower().isin(D_CRITICAL).astype(int).values
print(f"\n[D.2] Binary outcome target: critical = {D_CRITICAL}")
print(f"  Critical outcomes: {d_y_crit.sum():,} / {len(d_y_crit):,}  ({d_y_crit.mean()*100:.1f}%)")
for _v, _c in d_outcome_raw.str.lower().value_counts().items():
    _flag = " <- CRITICAL" if _v in D_CRITICAL else ""
    print(f"  {_v:20s}: {_c:6,}{_flag}")

In [ ]:
# ── StratifiedKFold(5) LightGBM OOF outcome-risk model ────────────────────
print("\n[D.3] StratifiedKFold(5) LightGBM outcome-risk model …")

d_skf       = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
d_oof_risk  = np.zeros(len(d_y_crit), dtype=float)
d_fold_mdls = []

d_lgb_params = dict(
    objective        = "binary",
    num_leaves       = 127,
    learning_rate    = 0.05,
    n_estimators     = 500,
    min_child_samples= 20,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    random_state     = RANDOM_STATE,
    verbose          = -1,
    n_jobs           = -1,
)

for _fold, (_tr_idx, _va_idx) in enumerate(d_skf.split(d_X, d_y_crit), 1):
    _X_tr, _X_va = d_X.iloc[_tr_idx], d_X.iloc[_va_idx]
    _y_tr, _y_va = d_y_crit[_tr_idx], d_y_crit[_va_idx]
    _mdl = lgb.LGBMClassifier(**d_lgb_params)
    _mdl.fit(
        _X_tr, _y_tr,
        eval_set=[(_X_va, _y_va)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
        categorical_feature=d_cat_features,
    )
    _proba = _mdl.predict_proba(_X_va)[:, 1]
    d_oof_risk[_va_idx] = _proba
    d_fold_mdls.append(_mdl)
    print(f"  Fold {_fold}: ROC-AUC={roc_auc_score(_y_va, _proba):.4f}  "
          f"best_iter={_mdl.best_iteration_}")

# OOF metrics (before calibration)
d_roc_raw  = roc_auc_score(d_y_crit, d_oof_risk)
d_pr_raw   = average_precision_score(d_y_crit, d_oof_risk)
d_brier_raw= brier_score_loss(d_y_crit, d_oof_risk)

def d_binary_ece(y_true, probs, n_bins=10):
    # Binary ECE helper for Pillar D.
    _bins = np.linspace(0, 1, n_bins + 1)
    _ece  = 0.0
    _n    = len(y_true)
    for _lo, _hi in zip(_bins[:-1], _bins[1:]):
        _mask = (probs >= _lo) & (probs < _hi)
        if _mask.sum() == 0:
            continue
        _ece += (_mask.sum() / _n) * abs(float(y_true[_mask].mean()) - float(probs[_mask].mean()))
    return float(_ece)

d_ece_raw = d_binary_ece(d_y_crit, d_oof_risk)

# Isotonic calibration
d_ir = IsotonicRegression(out_of_bounds="clip")
d_ir.fit(d_oof_risk, d_y_crit)
d_oof_risk_cal = np.clip(d_ir.transform(d_oof_risk), 0.0, 1.0)

d_ece_cal    = d_binary_ece(d_y_crit, d_oof_risk_cal)
d_brier_cal  = brier_score_loss(d_y_crit, d_oof_risk_cal)
d_roc_cal    = roc_auc_score(d_y_crit, d_oof_risk_cal)

print(f"\n  OOF ROC-AUC  : {d_roc_raw:.4f}  (expect ~0.78–0.88)")
print(f"  OOF PR-AUC   : {d_pr_raw:.4f}")
print(f"  OOF Brier    : {d_brier_raw:.4f}  (null={d_y_crit.mean()*(1-d_y_crit.mean()):.4f})")
print(f"  ECE before   : {d_ece_raw:.5f}")
print(f"  ECE after    : {d_ece_cal:.5f}   improvement={d_ece_raw - d_ece_cal:.5f}")

In [ ]:
# ── Reliability diagram: before vs after isotonic calibration ─────────────
fig, ax = plt.subplots(figsize=(7, 5))
_frac_b, _mean_b = calibration_curve(d_y_crit, d_oof_risk,     n_bins=10, strategy="uniform")
_frac_a, _mean_a = calibration_curve(d_y_crit, d_oof_risk_cal, n_bins=10, strategy="uniform")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
ax.plot(_mean_b, _frac_b, "o-", color="#1f77b4",
        label=f"Before (ECE={d_ece_raw:.4f})", lw=1.5)
ax.plot(_mean_a, _frac_a, "s-", color="#ff7f0e",
        label=f"After  (ECE={d_ece_cal:.4f})", lw=1.5)
ax.set_xlabel("Mean predicted risk", fontsize=11)
ax.set_ylabel("Fraction critical outcomes", fontsize=11)
ax.set_title("Pillar D — Outcome-Risk Model: Reliability Diagram\n"
             "Before vs After Isotonic Calibration", fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("pd_reliability.png", dpi=120)
plt.show()

# ── Feature importance (top 25, averaged across folds) ────────────────────
_imp_vals = np.mean([m.feature_importances_ for m in d_fold_mdls], axis=0)
d_imp_series = pd.Series(_imp_vals, index=d_feature_cols).sort_values(ascending=False)
_top25 = d_imp_series.head(25)
_colors_imp = ["#d62728" if f in VITALS else "#1f77b4" for f in _top25.index]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(_top25.index[::-1], _top25.values[::-1], color=_colors_imp[::-1])
ax.set_xlabel("Mean LightGBM feature importance (avg across folds)", fontsize=10)
ax.set_title("Pillar D — Outcome-Risk Model: Feature Importance\n"
             "(red=vital sign, blue=other; triage_acuity deliberately excluded)", fontsize=11)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("pd_feature_importance.png", dpi=120)
plt.show()
print("Figures saved: pd_reliability.png, pd_feature_importance.png")

In [ ]:
# ── Undertriage detection: flag L4/L5 patients with high predicted risk ───
print("\n[D.4] Outcome-anchored undertriage detection …")

d_risk = d_oof_risk_cal  # use calibrated risk throughout

# Threshold: 90th percentile of risk among L4/L5 patients
d_low_acuity_arr = d_acuity_true.isin([4, 5]).values      # numpy bool
d_risk_arr       = np.asarray(d_risk)                     # calibrated risk

d_risk_thresh    = float(np.percentile(d_risk_arr[d_low_acuity_arr], 90))
d_flagged_arr    = d_low_acuity_arr & (d_risk_arr >= d_risk_thresh)  # numpy bool
d_n_low          = int(d_low_acuity_arr.sum())
d_n_flagged      = int(d_flagged_arr.sum())

print(f"  90th-pct risk among L4/L5  : {d_risk_thresh:.4f}  <- threshold")
print(f"  L4/L5 patients             : {d_n_low:,}")
print(f"  Flagged undertriage cands  : {d_n_flagged:,}  ({d_n_flagged/d_n_low*100:.1f}% of L4/L5)")

# Validate: compare actual critical-outcome rates
d_flagged_crit_rate    = float(d_y_crit[d_flagged_arr].mean()) if d_n_flagged > 0 else float("nan")
d_nonflag_low_arr      = d_low_acuity_arr & ~d_flagged_arr
d_nonflag_crit_rate    = float(d_y_crit[d_nonflag_low_arr].mean())
d_overall_crit_rate    = float(d_y_crit.mean())

print(f"\n  VALIDATION — Actual critical-outcome rates:")
print(f"    Flagged L4/L5             : {d_flagged_crit_rate*100:.1f}%  <- enriched for critical outcomes")
print(f"    Non-flagged L4/L5         : {d_nonflag_crit_rate*100:.1f}%  <- baseline")
print(f"    Overall population        : {d_overall_crit_rate*100:.1f}%")
print(f"\n  Relative risk (flagged vs non-flagged L4/L5): "
      f"{d_flagged_crit_rate / max(d_nonflag_crit_rate, 1e-9):.1f}x")

print(f"\n  Flagged candidates — disposition breakdown:")
_flag_disp = d_outcome_raw.iloc[np.where(d_flagged_arr)[0]].str.lower().value_counts()
for _disp, _cnt in _flag_disp.items():
    print(f"    {_disp:20s}: {_cnt:4d} ({_cnt/max(d_n_flagged,1)*100:.1f}%)")

In [ ]:
# ── Bar chart: actual critical rate — flagged vs non-flagged L4/L5 vs overall ──
fig, ax = plt.subplots(figsize=(7, 4))
_groups = ["Flagged L4/L5\n(undertriage\ncandidates)", "Non-flagged\nL4/L5", "Overall\npopulation"]
_rates  = [d_flagged_crit_rate * 100, d_nonflag_crit_rate * 100, d_overall_crit_rate * 100]
_cols   = ["#d62728", "#1f77b4", "#2ca02c"]
_bars   = ax.bar(_groups, _rates, color=_cols, width=0.55, edgecolor="white")
for _bar, _rate in zip(_bars, _rates):
    _lbl  = f"{_rate:.1f}%" if not np.isnan(_rate) else "N/A"
    _ypos = _bar.get_height() + 0.5 if not np.isnan(_bar.get_height()) else 0.5
    ax.text(_bar.get_x() + _bar.get_width() / 2, _ypos, _lbl,
            ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_ylabel("Actual critical-outcome rate (%)", fontsize=11)
ax.set_title("Pillar D — Undertriage Detector Validation\n"
             "Flagged L4/L5 patients have higher actual critical-outcome rate\n"
             "(critical = admitted | transferred | deceased)", fontsize=10)
_max_r = max(r for r in _rates if not np.isnan(r))
ax.set_ylim(0, _max_r * 1.3)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("pd_undertriage_validation.png", dpi=120)
plt.show()
print("Figure saved: pd_undertriage_validation.png")

In [ ]:
# ── Equity audit on undertriage flags (bootstrap 95% CIs) ────────────────
print("\n[D.5] Equity audit — flag rate by protected attribute (L4/L5 patients only) …")

_low_idx  = np.where(d_low_acuity_arr)[0]
d_equity_df = d_train.iloc[_low_idx].copy().reset_index(drop=True)
d_equity_df["flagged"]  = d_flagged_arr[_low_idx].astype(float)
d_equity_df["critical"] = d_y_crit[_low_idx]

def _d_bootstrap_ci(values, n_boot=2000, seed=42, ci=0.95):
    # Bootstrap CI for mean (Pillar D internal helper).
    _rng   = np.random.default_rng(seed)
    _boots = np.array([_rng.choice(values, size=len(values), replace=True).mean()
                       for _ in range(n_boot)])
    _alpha = (1 - ci) / 2
    return float(np.quantile(_boots, _alpha)), float(np.quantile(_boots, 1 - _alpha))

d_equity_results = {}
_overall_flag_rate = float(d_equity_df["flagged"].mean())

for attr in PROTECTED:
    print(f"\n  -- {attr} (overall L4/L5 flag rate = {_overall_flag_rate:.4f}) --")
    _rows = []
    for grp, sub in d_equity_df.groupby(attr, observed=True):
        _fv = sub["flagged"].astype(float).values
        if len(_fv) < 10:
            continue
        _mean = float(_fv.mean())
        _lo, _hi = _d_bootstrap_ci(_fv)
        _straddles = bool(_lo <= _overall_flag_rate <= _hi)
        _rows.append({"group": str(grp), "n": len(_fv),
                      "flag_rate": round(_mean, 4),
                      "ci_low": round(_lo, 4), "ci_high": round(_hi, 4),
                      "ci_straddles_overall": _straddles})
    _tbl = pd.DataFrame(_rows)
    d_equity_results[attr] = _tbl
    print(_tbl[["group","n","flag_rate","ci_low","ci_high","ci_straddles_overall"]].to_string(index=False))

In [ ]:
# ── Subgroup calibration: ECE within language and age_group ───────────────
print("\n[D.6] Subgroup calibration (ECE within each language and age_group) …")

d_subgroup_ece = {}

for attr in ["language", "age_group"]:
    print(f"\n  -- {attr} --")
    _rows = []
    for grp, sub in d_train.groupby(attr, observed=True):
        _idx  = sub.index.values
        _sr   = d_risk_arr[_idx]
        _sy   = d_y_crit[_idx]
        if _sy.sum() < 5 or len(_sy) < 30:
            continue
        _ece = d_binary_ece(_sy, _sr)
        _rows.append({"group": str(grp), "n": len(_sy),
                      "n_crit": int(_sy.sum()),
                      "prev_pct": round(float(_sy.mean()) * 100, 1),
                      "ECE": round(_ece, 5)})
        print(f"    {str(grp):20s}  n={len(_sy):6,}  prev={_sy.mean()*100:4.1f}%  ECE={_ece:.5f}")
    d_subgroup_ece[attr] = _rows

# Subgroup ECE bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for _ax, attr in zip(axes, ["language", "age_group"]):
    _rows = d_subgroup_ece.get(attr, [])
    if not _rows:
        _ax.axis("off")
        continue
    _grps = [r["group"] for r in _rows]
    _eces = [r["ECE"]   for r in _rows]
    _ax.bar(range(len(_grps)), _eces, color="#4C72B0", edgecolor="white")
    _ax.axhline(d_ece_cal, color="#d62728", linestyle="--", linewidth=1.5,
                label=f"Overall ECE={d_ece_cal:.5f}")
    _ax.set_xticks(range(len(_grps)))
    _ax.set_xticklabels(_grps, rotation=35, ha="right", fontsize=9)
    _ax.set_ylabel("ECE", fontsize=10)
    _ax.set_title(f"Subgroup Calibration — {attr}", fontsize=11)
    _ax.legend(fontsize=8)
    _ax.grid(axis="y", alpha=0.3)
fig.suptitle("Pillar D — Outcome-Risk Model: ECE Within Subgroups\n"
             "(similar ECE across groups → fair calibration)", fontsize=11)
fig.tight_layout()
plt.savefig("pd_subgroup_ece.png", dpi=120)
plt.show()
print("Figure saved: pd_subgroup_ece.png")

In [ ]:
# ── Data forensics: MAP / pulse_pressure / shock_index are exact formulas ─
print("\n[D.7] Data forensics — verifying derived vital signs are exact formulae …")
from sklearn.linear_model import LinearRegression as _LR

_df_for = d_train[
    ["systolic_bp", "diastolic_bp", "mean_arterial_pressure",
     "pulse_pressure", "heart_rate", "shock_index"]
].dropna()
print(f"  Complete-case rows: {len(_df_for):,}")

# (a) MAP = (SBP + 2*DBP)/3
_X_map = _df_for[["systolic_bp", "diastolic_bp"]].values
_y_map = _df_for["mean_arterial_pressure"].values
_lr_map = _LR().fit(_X_map, _y_map)
_pred_map = _lr_map.predict(_X_map)
d_r2_map  = float(1 - np.sum((_y_map - _pred_map)**2) / np.sum((_y_map - _y_map.mean())**2))
_std_map  = float(np.std(_y_map - _pred_map))
print(f"\n  MAP ~ SBP + DBP:    R²={d_r2_map:.6f}  resid_std={_std_map:.4f}")
print(f"    coef: SBP={_lr_map.coef_[0]:.4f}  DBP={_lr_map.coef_[1]:.4f}  "
      f"intercept={_lr_map.intercept_:.4f}")
print(f"    [Expected: ~0.333 SBP + 0.667 DBP + 0]")

# (b) Pulse pressure = SBP - DBP
_pp_computed = _df_for["systolic_bp"].values - _df_for["diastolic_bp"].values
_pp_actual   = _df_for["pulse_pressure"].values
d_r2_pp      = float(1 - np.sum((_pp_actual - _pp_computed)**2) /
                     np.sum((_pp_actual - _pp_actual.mean())**2))
_std_pp      = float(np.std(_pp_actual - _pp_computed))
print(f"\n  Pulse pressure = SBP - DBP:   R²={d_r2_pp:.6f}  resid_std={_std_pp:.6f}")

# (c) Shock index = HR / SBP
_si_computed = _df_for["heart_rate"].values / _df_for["systolic_bp"].values
_si_actual   = _df_for["shock_index"].values
d_r2_si      = float(1 - np.sum((_si_actual - _si_computed)**2) /
                     np.sum((_si_actual - _si_actual.mean())**2))
_std_si      = float(np.std(_si_actual - _si_computed))
print(f"\n  Shock index = HR / SBP:       R²={d_r2_si:.6f}  resid_std={_std_si:.6f}")

print(f"\n  Conclusion: R²≈1.0 for all three → generator computed these by formula (no noise).")
print(f"  In REAL clinical data, R² would be <0.99 due to equipment variation and timing.")
print(f"\n  Forensics summary:")
print(f"  {'Derived variable':<28} {'R²':>10}  {'resid_std':>12}  formula")
print(f"  {'─'*28}  {'─'*10}  {'─'*12}  {'─'*30}")
print(f"  {'mean_arterial_pressure':<28} {d_r2_map:>10.6f}  {_std_map:>12.4f}  (SBP + 2*DBP)/3")
print(f"  {'pulse_pressure':<28} {d_r2_pp:>10.6f}  {_std_pp:>12.6f}  SBP - DBP")
print(f"  {'shock_index':<28} {d_r2_si:>10.6f}  {_std_si:>12.6f}  HR / SBP")

In [ ]:
# ── triage_second_opinion() demo on 5 hand-picked case studies ────────────
print("\n[D.8] Second-opinion decision-support demo …")

# Red-flag regex patterns (subset from Pillar B)
_D_RF_PATTERNS = [
    r"thunderclap\s+headache",
    r"chest\s+pain.{0,30}(diaphoresis|sweat)",
    r"(diaphoresis|sweat).{0,30}chest\s+pain",
    r"stroke|facial\s+droop|arm\s+weak|aphasia",
    r"sepsis|rigors|bacteraemia",
    r"cardiac\s+arrest|pulseless",
    r"respiratory\s+fail",
    r"unresponsive",
    r"haemorrhage|hemorrhage",
    r"(severe|acute|crushing)\s+pain",
    r"shortness\s+of\s+breath.{0,30}severe",
    r"syncope|loss\s+of\s+consciousness",
]
_d_rf_re = [_re.compile(p, _re.IGNORECASE) for p in _D_RF_PATTERNS]


def _d_red_flag_hits(text):
    if not isinstance(text, str):
        return []
    return [pat.pattern for pat in _d_rf_re if pat.search(text)]


def _d_risk_tier(prob):
    if prob >= 0.70:
        return "CRITICAL (>=70%)"
    elif prob >= 0.45:
        return "HIGH (45-70%)"
    elif prob >= 0.25:
        return "MODERATE (25-45%)"
    elif prob >= 0.10:
        return "LOW-MODERATE (10-25%)"
    return "LOW (<10%)"


def triage_second_opinion(patient_row_raw, calibrated_risk_prob):
    # Decision-support second opinion for a triage patient.
    # Inputs:
    #   patient_row_raw       - pandas Series (row from train_df with acuity + text)
    #   calibrated_risk_prob  - float, OOF calibrated P(critical outcome)
    # Returns dict with: assigned_acuity, independent_outcome_risk_pct, risk_tier,
    #   red_flag_text_hits, undertriage_warning, warning_reasons.
    acuity    = int(patient_row_raw.get(TARGET, -1))
    risk_prob = float(calibrated_risk_prob)
    raw_text  = str(patient_row_raw.get("chief_complaint_raw", ""))
    rf_hits   = _d_red_flag_hits(raw_text)

    undertriage = False
    reasons     = []

    if acuity in [4, 5] and risk_prob >= d_risk_thresh:
        undertriage = True
        reasons.append(
            f"Assigned L{acuity} but independent outcome risk ({risk_prob*100:.1f}%) "
            f">= 90th-pct L4/L5 risk threshold ({d_risk_thresh*100:.1f}%) — consider uptriage"
        )
    if rf_hits:
        reasons.append(f"Red-flag text patterns: {', '.join(rf_hits[:3])}")
        if acuity in [3, 4, 5]:
            undertriage = True

    return {
        "patient_id"                  : str(patient_row_raw.get("patient_id", "N/A")),
        "assigned_acuity"             : acuity,
        "actual_disposition"          : str(patient_row_raw.get("disposition", "unknown")),
        "independent_outcome_risk_pct": round(risk_prob * 100, 1),
        "risk_tier"                   : _d_risk_tier(risk_prob),
        "chief_complaint_raw"         : raw_text[:120],
        "red_flag_text_hits"          : rf_hits,
        "undertriage_warning"         : undertriage,
        "warning_reasons"             : reasons,
    }


def _print_case(cdict, label):
    sep = "-" * 62
    print(f"\n  {sep}")
    print(f"  CASE: {label}")
    print(f"  {sep}")
    print(f"  Patient ID      : {cdict['patient_id']}")
    print(f"  Assigned acuity : L{cdict['assigned_acuity']}")
    print(f"  Actual outcome  : {cdict['actual_disposition']}")
    print(f"  Independent risk: {cdict['independent_outcome_risk_pct']}%  [{cdict['risk_tier']}]")
    print(f"  Chief complaint : \"{cdict['chief_complaint_raw']}\"")
    if cdict["red_flag_text_hits"]:
        print(f"  RED FLAGS FOUND : {cdict['red_flag_text_hits']}")
    else:
        print(f"  Red flags       : none")
    if cdict["undertriage_warning"]:
        print(f"  *** UNDERTRIAGE WARNING ***")
        for r in cdict["warning_reasons"]:
            print(f"    -> {r}")
    else:
        print(f"  Assessment      : consistent with outcome risk")
    print(f"  {sep}")

# Build demo dataframe (reset index to align with d_risk_arr)
_demo_df = d_train.copy().reset_index(drop=True)
_demo_df["_risk"]    = d_risk_arr
_demo_df["_flagged"] = d_flagged_arr

# 5 case studies (using reset-index positions)
_case_a = _demo_df[_demo_df[TARGET] == 1]["_risk"].idxmax()
_case_b = _demo_df[_demo_df[TARGET] == 5]["_risk"].idxmin()
_flagged_cands = _demo_df[_demo_df["_flagged"]].sort_values("_risk", ascending=False)
_case_c = int(_flagged_cands.iloc[0].name)
_case_d = int(_flagged_cands.iloc[min(1, len(_flagged_cands)-1)].name)
_r75 = float(np.percentile(d_risk_arr[d_low_acuity_arr], 75))
_near_miss = _demo_df[(_demo_df[TARGET] == 3) & (_demo_df["_risk"] >= _r75)].sort_values("_risk", ascending=False)
_case_e = int(_near_miss.iloc[0].name) if len(_near_miss) > 0 else int(_flagged_cands.iloc[2 if len(_flagged_cands)>2 else -1].name)

_case_studies = {
    "A - Clear L1 (highest risk among L1 patients)"  : _case_a,
    "B - Clear L5 (lowest risk among L5 patients)"   : _case_b,
    "C - FLAGGED undertriage (L4/5, highest risk #1)": _case_c,
    "D - FLAGGED undertriage (L4/5, highest risk #2)": _case_d,
    "E - Near-miss (L3, risk > 75th-pct of L4/L5)"  : _case_e,
}

print("\n  === SECOND-OPINION CASE STUDIES ===")
for _lbl, _idx in _case_studies.items():
    _row  = _demo_df.iloc[_idx]
    _prob = float(d_risk_arr[_idx])
    _d    = triage_second_opinion(_row, _prob)
    _print_case(_d, _lbl)

In [ ]:
# ── Pillar D headline summary ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("  PILLAR D — HEADLINE NUMBERS")
print("=" * 70)
print(f"  Outcome ROC-AUC (OOF)    : {d_roc_raw:.4f}  (calibrated: {d_roc_cal:.4f})")
print(f"  Outcome PR-AUC           : {d_pr_raw:.4f}")
print(f"  Brier score (before/after): {d_brier_raw:.4f} -> {d_brier_cal:.4f}")
print(f"  ECE before -> after      : {d_ece_raw:.5f} -> {d_ece_cal:.5f}")
print(f"  Undertriage flagged      : {d_n_flagged:,} / {d_n_low:,} L4/L5 ({d_n_flagged/d_n_low*100:.1f}%)")
print(f"  Flagged crit. rate       : {d_flagged_crit_rate*100:.1f}% vs {d_nonflag_crit_rate*100:.1f}% non-flagged")
print(f"  Relative risk            : {d_flagged_crit_rate / max(d_nonflag_crit_rate, 1e-9):.1f}x")
print(f"  MAP forensics R²         : {d_r2_map:.6f}")
print(f"  Pulse pressure R²        : {d_r2_pp:.6f}")
print(f"  Shock index R²           : {d_r2_si:.6f}")
print("=" * 70)

---
## Section 7 — Limitations, External Validity & Reproducibility

### Synthetic-Data Limitations

The dataset is entirely synthetic (no patient health information). The key limitations are:

1. **Deterministic label:** `triage_acuity` is a near-deterministic physiological function
   (NEWS2 + GCS + pain + shock index). Real triage involves human judgment, gestalt assessment,
   and implicit bias — all absent here. Model accuracy is therefore an upper bound, not a
   realistic benchmark.

2. **Text AUC = 1.0 artefact:** The chief complaint generator embeds severity adjectives
   ("severe", "acute", "mild") that directly encode the label. In real free-text chief
   complaints, text-only AUC would be ~0.65–0.75 (MIMIC-IV-ED estimate).

3. **No inter-rater noise:** Fifty nurses assign triage acuity with near-zero variability
   (residual std 0.014). Real ESI weighted κ ranges 0.71–0.91; real audit tools would flag
   multiple outlier nurses and reveal nurse-specific bias.

4. **No demographic bias:** The synthetic policy is age-blind and demographically neutral.
   Real ED data carries documented disparities (OR 0.76–1.49 across protected attributes).
   Our toolkit's null result on this data is a **positive property** (it validates the
   negative control), not a finding about real EDs.

5. **Pillar D undertriage enrichment is modest (1.3×):** The outcome model's relative risk
   for flagged L4/L5 patients is only 1.3× in this synthetic dataset because the acuity label
   is near-deterministic — L4/L5 patients genuinely have low outcome risk by construction.
   In real EDs, where undertriage reflects human judgment errors and presentation complexity,
   outcome-based detectors routinely yield 2–5× enrichment. The 1.3× finding correctly reflects
   the synthetic data's structure, not a weakness of the detection framework.

### External Validation Pathway

| Dataset | Why relevant | Access |
|---|---|---|
| **MIMIC-IV-ED** (~425k stays) | Identical triage schema, real free-text chief complaints, real acuity labels, real demographics | PhysioNet credentialed access |
| **NHAMCS** (nationally representative) | Includes insurance, language, age — directly maps to our equity audit attributes | CDC public use files |

The four audit functions (`audit_by_group`, `inject_undertriage`, `audit_inter_rater`,
`build_literature_contrast`) can be applied to either dataset **without modification**,
provided `triage_acuity`, `news2_score`, and `triage_nurse_id` columns are present.

### Reproducibility

| Component | Detail |
|---|---|
| Random seed | 42 everywhere (model, bootstrap, train/test split, SHAP subsample) |
| Python version | 3.11 (Kaggle standard image) |
| Key library versions | LightGBM ≥ 3.3, scikit-learn ≥ 1.0, shap ≥ 0.41, scipy ≥ 1.7 |
| Kaggle kernel | CPU-only, no GPU, no internet; runtime ~15–18 min |
| Data path | `/kaggle/input/competitions/triagegeist/` (note `competitions/` segment) |
| Leakage check | `disposition`, `ed_los_hours` never appear in `X` or `X_text` |

---
## Section 8 — Conclusion & Clinical Recommendations

### What a hospital clinical-AI team should take away

**1. Calibration before deployment.**
An accuracy number is not enough. This notebook demonstrates that raw LightGBM probabilities
have ECE = 0.0067 — off enough to mislead a nurse about a patient's true risk. Isotonic
recalibration cuts this to 0.0014. Any model entering a clinical decision-support tool must
pass a calibration audit before go-live, and must be recalibrated on local hospital data.

**2. The age-blindness finding demands local replication.**
On this synthetic dataset the model assigns zero importance to age, BMI, weight, and height —
the policy is purely physiological. In real EDs, geriatric undertriage exceeds 22%. A hospital
deploying this model on real data should run the SHAP audit on their local population: if age
importance rises above noise, the model has absorbed a real-world bias that must be corrected.

**3. The NLP flagger's value is independent of its AUC.**
In synthetic data the text model achieves AUC ≈ 1.0 as an artefact. In real data, the value
proposition is different: the red-flag lexicon fires on 13.4% of presentations with 100%
high-acuity concentration for 8 patterns (thunderclap headache, cardiac arrest, anaphylaxis,
aortic dissection, acute abdomen, meningeal signs, DKA, respiratory failure). These are
non-discretionary ESI L1 triggers. A deployed lexicon scanner operating in the background
as a nurse completes a subjective assessment provides a last-resort catch at essentially
zero cost.

**4. The audit toolkit is the impact pathway.**
The four functions in Pillar C — `audit_by_group`, `inject_undertriage`, `audit_inter_rater`,
and the literature contrast builder — constitute a reusable equity audit toolkit. Applied to
MIMIC-IV-ED or a hospital's own triage logs, they would (a) detect whether protected-attribute
disparities exist, (b) quantify their magnitude in clinically interpretable NEWS2-residual
units, (c) flag individual nurses for targeted re-training, and (d) benchmark against the
published literature. The toolkit's positive-control validation confirms a detection threshold
of approximately 0.05 acuity units — smaller than any effect size documented in the literature.

---

### Summary of Validated Headline Numbers

| Metric | Value |
|---|---|
| OOF Accuracy | **0.855** |
| Macro-F1 | **0.870** |
| Quadratic Weighted Kappa | **0.930** |
| Safety recall L1 (resuscitation) | **0.920** |
| Safety recall L2 (emergent) | **0.969** |
| ECE before → after isotonic | **0.00673 → 0.00144** |
| Text-only ROC-AUC | 1.000 (synthetic artefact — see note) |
| Vitals-only ROC-AUC | 0.997 |
| Lexicon flagged / precision / recall | 13.4% / 0.40 / 0.26 |
| Negative control: all bias CIs ∋ 0 | True (null result, expected) |
| Positive control detection threshold | 0.05 acuity units (frac×delta) |
| Inter-rater outlier nurses | 1 / 50 (NURSE-0013, residual +0.035) |
| **Pillar D — Outcome-risk ROC-AUC** | **0.813** (triage_acuity excluded) |
| Pillar D — PR-AUC | 0.729 |
| Pillar D — Brier score | 0.166 (null: 0.235) |
| Pillar D — ECE before → after isotonic | 0.0145 → ~0.000 |
| Pillar D — Undertriage flags | 3,646 (10.6% of L4/L5 patients) |
| Pillar D — Flagged critical rate | 11.9% vs 9.3% non-flagged (1.3× RR) |
| Pillar D — Derived-vital R² | ≥ 0.9999 (MAP, pulse_pressure, shock_index) |
| Pillar D — Subgroup ECE range | 0.001–0.015 (language + age_group subgroups) |

*Kernel:* `fairlanderflick/triagegeist-ed-triage-toolkit` | *Seed:* 42 | *CPU-only, no internet*